In [ ]:
"""
Optimized Order Flow Imbalance (OFI) computation.

Improvements:
- Fully vectorized (no Python loops)
- Minimal memory copies (copy=False where possible)
- Pre-extracted columns for multi-level OFI
- Optional float32 support for memory efficiency
- No unnecessary DataFrame copies
"""

import numpy as np
import pandas as pd


# =========================
# Core OFI computation
# =========================
def _level_ofi_arrays(
    bp: np.ndarray,
    bs: np.ndarray,
    ap: np.ndarray,
    as_: np.ndarray,
    dtype=np.float64,
) -> np.ndarray:
    """Vectorized single-level OFI on NumPy arrays."""

    # Ensure correct dtype without unnecessary copies
    bp = np.asarray(bp, dtype=dtype)
    bs = np.asarray(bs, dtype=dtype)
    ap = np.asarray(ap, dtype=dtype)
    as_ = np.asarray(as_, dtype=dtype)

    n = len(bp)
    ofi = np.zeros(n, dtype=dtype)

    # Precompute comparisons
    bp_up = bp[1:] > bp[:-1]
    bp_down = bp[1:] < bp[:-1]

    ap_down = ap[1:] < ap[:-1]
    ap_up = ap[1:] > ap[:-1]

    # Compute deltas
    delta_bid = np.where(
        bp_up,
        bs[1:],
        np.where(bp_down, -bs[:-1], bs[1:] - bs[:-1]),
    )

    delta_ask = np.where(
        ap_down,
        -as_[1:],
        np.where(ap_up, as_[:-1], as_[1:] - as_[:-1]),
    )

    ofi[1:] = delta_bid - delta_ask
    return ofi


# =========================
# Single-level OFI
# =========================
def compute_single_level_ofi(
    df: pd.DataFrame,
    dtype=np.float64,
) -> pd.Series:
    """Compute Level-1 OFI (vectorized)."""

    bp = df["bid_price_1"].to_numpy(dtype=dtype, copy=False)
    bs = df["bid_size_1"].to_numpy(dtype=dtype, copy=False)
    ap = df["ask_price_1"].to_numpy(dtype=dtype, copy=False)
    as_ = df["ask_size_1"].to_numpy(dtype=dtype, copy=False)

    ofi = _level_ofi_arrays(bp, bs, ap, as_, dtype=dtype)
    return pd.Series(ofi, index=df.index, name="ofi_1")


# =========================
# Multi-level OFI
# =========================
def compute_multi_level_ofi(
    df: pd.DataFrame,
    max_level: int = 5,
    dtype=np.float64,
) -> pd.DataFrame:
    """
    Compute OFI for multiple levels efficiently.
    Returns:
        ofi_1 ... ofi_k
        ofi_cumul_1 ... ofi_cumul_k
    """

    n = len(df)

    # Pre-extract all columns (avoid repeated Pandas access)
    bid_prices = [
        df[f"bid_price_{i}"].to_numpy(dtype=dtype, copy=False)
        for i in range(1, max_level + 1)
    ]
    bid_sizes = [
        df[f"bid_size_{i}"].to_numpy(dtype=dtype, copy=False)
        for i in range(1, max_level + 1)
    ]
    ask_prices = [
        df[f"ask_price_{i}"].to_numpy(dtype=dtype, copy=False)
        for i in range(1, max_level + 1)
    ]
    ask_sizes = [
        df[f"ask_size_{i}"].to_numpy(dtype=dtype, copy=False)
        for i in range(1, max_level + 1)
    ]

    # Preallocate output (more memory efficient than dict + copies)
    out = np.zeros((n, max_level * 2), dtype=dtype)

    cumul = np.zeros(n, dtype=dtype)

    for lvl in range(max_level):
        ofi_lvl = _level_ofi_arrays(
            bid_prices[lvl],
            bid_sizes[lvl],
            ask_prices[lvl],
            ask_sizes[lvl],
            dtype=dtype,
        )

        # Store OFI
        out[:, lvl] = ofi_lvl

        # Update cumulative
        cumul += ofi_lvl
        out[:, max_level + lvl] = cumul

    # Column names
    columns = (
        [f"ofi_{i}" for i in range(1, max_level + 1)]
        + [f"ofi_cumul_{i}" for i in range(1, max_level + 1)]
    )

    return pd.DataFrame(out, index=df.index, columns=columns)


# =========================
# Aggregation to time bars
# =========================
def aggregate_ofi_to_time_bars(
    ofi_df: pd.DataFrame,
    timestamps: pd.Series,
    bar_size: str = "1s",
) -> pd.DataFrame:
    """Aggregate OFI into time bars (no unnecessary copies)."""

    return (
        ofi_df
        .set_index(pd.Index(timestamps, name="datetime"))
        .resample(bar_size)
        .sum()
    )

In [ ]:
"""
Optimized Microstructure feature extraction.

Key improvements:
- NumPy-first (minimal Pandas overhead)
- No repeated column access
- No unnecessary astype copies
- Shared computations (mid-price reused)
- Preallocation instead of Python loops
"""

import numpy as np
import pandas as pd


# =========================
# Core helpers
# =========================
def _get_array(df, col, dtype):
    return df[col].to_numpy(dtype=dtype, copy=False)


# =========================
# Feature computations
# =========================
def compute_mid_price(df: pd.DataFrame, dtype=np.float64) -> pd.Series:
    bp = _get_array(df, "bid_price_1", dtype)
    ap = _get_array(df, "ask_price_1", dtype)
    mid = (ap + bp) * 0.5
    return pd.Series(mid, index=df.index, name="mid_price")


def compute_spread(df: pd.DataFrame, dtype=np.float64) -> pd.Series:
    bp = _get_array(df, "bid_price_1", dtype)
    ap = _get_array(df, "ask_price_1", dtype)
    return pd.Series(ap - bp, index=df.index, name="spread")


def compute_volume_imbalance(df: pd.DataFrame, dtype=np.float64) -> pd.Series:
    bid_s = _get_array(df, "bid_size_1", dtype)
    ask_s = _get_array(df, "ask_size_1", dtype)

    total = bid_s + ask_s
    out = np.zeros_like(total, dtype=dtype)

    mask = total != 0
    out[mask] = (bid_s[mask] - ask_s[mask]) / total[mask]

    return pd.Series(out, index=df.index, name="volume_imbalance")


def compute_weighted_mid_price(df: pd.DataFrame, dtype=np.float64) -> pd.Series:
    bp = _get_array(df, "bid_price_1", dtype)
    ap = _get_array(df, "ask_price_1", dtype)
    bid_s = _get_array(df, "bid_size_1", dtype)
    ask_s = _get_array(df, "ask_size_1", dtype)

    total = bid_s + ask_s
    out = np.empty_like(total, dtype=dtype)

    mask = total != 0
    out[mask] = (ap[mask] * bid_s[mask] + bp[mask] * ask_s[mask]) / total[mask]

    # fallback to mid-price where division invalid
    mid = (ap + bp) * 0.5
    out[~mask] = mid[~mask]

    return pd.Series(out, index=df.index, name="weighted_mid_price")


def compute_book_depth(df: pd.DataFrame, max_level: int = 5, dtype=np.float64) -> pd.DataFrame:
    n = len(df)

    bid_depth = np.zeros(n, dtype=dtype)
    ask_depth = np.zeros(n, dtype=dtype)

    # accumulate using NumPy arrays (faster than Pandas sum)
    for i in range(1, max_level + 1):
        bid_depth += _get_array(df, f"bid_size_{i}", dtype)
        ask_depth += _get_array(df, f"ask_size_{i}", dtype)

    total_depth = bid_depth + ask_depth

    return pd.DataFrame(
        {
            "bid_depth": bid_depth,
            "ask_depth": ask_depth,
            "total_depth": total_depth,
        },
        index=df.index,
    )


def compute_depth_imbalance(df: pd.DataFrame, max_level: int = 5, dtype=np.float64) -> pd.Series:
    depth = compute_book_depth(df, max_level, dtype)

    bid_d = depth["bid_depth"].to_numpy(copy=False)
    ask_d = depth["ask_depth"].to_numpy(copy=False)
    total_d = depth["total_depth"].to_numpy(copy=False)

    out = np.zeros_like(total_d, dtype=dtype)
    mask = total_d != 0
    out[mask] = (bid_d[mask] - ask_d[mask]) / total_d[mask]

    return pd.Series(out, index=df.index, name="depth_imbalance")


def compute_price_distances(df: pd.DataFrame, max_level: int = 5, dtype=np.float64) -> pd.DataFrame:
    bp1 = _get_array(df, "bid_price_1", dtype)
    ap1 = _get_array(df, "ask_price_1", dtype)
    mid = (ap1 + bp1) * 0.5

    n = len(df)
    out = np.zeros((n, max_level * 2), dtype=dtype)

    for lvl in range(1, max_level + 1):
        ap = _get_array(df, f"ask_price_{lvl}", dtype)
        bp = _get_array(df, f"bid_price_{lvl}", dtype)

        out[:, 2 * (lvl - 1)] = ap - mid
        out[:, 2 * (lvl - 1) + 1] = mid - bp

    columns = []
    for lvl in range(1, max_level + 1):
        columns.append(f"ask_dist_{lvl}")
        columns.append(f"bid_dist_{lvl}")

    return pd.DataFrame(out, index=df.index, columns=columns)


def compute_return(df: pd.DataFrame, horizon: int = 1, dtype=np.float64) -> pd.Series:
    bp = _get_array(df, "bid_price_1", dtype)
    ap = _get_array(df, "ask_price_1", dtype)

    mid = (ap + bp) * 0.5
    log_mid = np.log(mid)

    ret = np.empty_like(log_mid)
    ret[:-horizon] = log_mid[horizon:] - log_mid[:-horizon]
    ret[-horizon:] = np.nan

    return pd.Series(ret, index=df.index, name=f"return_{horizon}")


# =========================
# Combined pipeline
# =========================
def compute_all_features(df: pd.DataFrame, max_level: int = 5, dtype=np.float64) -> pd.DataFrame:
    n = len(df)

    # Pre-extract core arrays once
    bp = _get_array(df, "bid_price_1", dtype)
    ap = _get_array(df, "ask_price_1", dtype)
    bid_s = _get_array(df, "bid_size_1", dtype)
    ask_s = _get_array(df, "ask_size_1", dtype)

    mid = (ap + bp) * 0.5
    spread = ap - bp

    # Volume imbalance
    total_vol = bid_s + ask_s
    vi = np.zeros(n, dtype=dtype)
    mask = total_vol != 0
    vi[mask] = (bid_s[mask] - ask_s[mask]) / total_vol[mask]

    # Weighted mid
    wmp = np.empty(n, dtype=dtype)
    wmp[mask] = (ap[mask] * bid_s[mask] + bp[mask] * ask_s[mask]) / total_vol[mask]
    wmp[~mask] = mid[~mask]

    # Depth
    bid_depth = np.zeros(n, dtype=dtype)
    ask_depth = np.zeros(n, dtype=dtype)

    for i in range(1, max_level + 1):
        bid_depth += _get_array(df, f"bid_size_{i}", dtype)
        ask_depth += _get_array(df, f"ask_size_{i}", dtype)

    total_depth = bid_depth + ask_depth

    # Depth imbalance
    di = np.zeros(n, dtype=dtype)
    mask_d = total_depth != 0
    di[mask_d] = (bid_depth[mask_d] - ask_depth[mask_d]) / total_depth[mask_d]

    return pd.DataFrame(
        {
            "mid_price": mid,
            "spread": spread,
            "volume_imbalance": vi,
            "weighted_mid_price": wmp,
            "bid_depth": bid_depth,
            "ask_depth": ask_depth,
            "total_depth": total_depth,
            "depth_imbalance": di,
        },
        index=df.index,
    )

In [ ]:
"""
Optimized multi-horizon label generation for LOB mid-price prediction.

Key improvements:
- Single mid-price computation
- Minimal memory allocations
- Vectorized shifting
- Efficient smoothing via cumsum
- Reused buffers
"""

import numpy as np
import pandas as pd
from typing import List


DEFAULT_HORIZONS: List[int] = [10, 20, 50, 100]


# =========================
# Core helpers
# =========================
def _mid_price_array(df: pd.DataFrame, dtype=np.float64) -> np.ndarray:
    bp = df["bid_price_1"].to_numpy(dtype=dtype, copy=False)
    ap = df["ask_price_1"].to_numpy(dtype=dtype, copy=False)
    return (ap + bp) * 0.5


def _future_shift(arr: np.ndarray, k: int) -> np.ndarray:
    """Efficient forward shift with NaN tail."""
    n = len(arr)
    out = np.full(n, np.nan, dtype=arr.dtype)
    if k < n:
        out[: n - k] = arr[k:]
    return out


def _smoothed_mid(m: np.ndarray, k: int) -> np.ndarray:
    """Efficient moving average using cumsum."""
    n = len(m)
    out = np.full(n, np.nan, dtype=m.dtype)

    if k >= n:
        return out

    cumsum = np.cumsum(m, dtype=m.dtype)

    # sum[t+1:t+k+1] = cumsum[t+k] - cumsum[t]
    out[: n - k] = (cumsum[k:] - cumsum[:-k]) / k
    return out


# =========================
# Regression labels
# =========================
def make_regression_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    dtype=np.float64,
) -> pd.DataFrame:

    if horizons is None:
        horizons = DEFAULT_HORIZONS

    m = _mid_price_array(df, dtype)
    n = len(m)

    out = np.empty((n, len(horizons)), dtype=dtype)

    for i, k in enumerate(horizons):
        shifted = _future_shift(m, k)
        out[:, i] = shifted - m

    cols = [f"delta_mid_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


def make_return_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    dtype=np.float64,
) -> pd.DataFrame:

    if horizons is None:
        horizons = DEFAULT_HORIZONS

    m = _mid_price_array(df, dtype)
    log_m = np.log(m)

    n = len(m)
    out = np.empty((n, len(horizons)), dtype=dtype)

    for i, k in enumerate(horizons):
        shifted = _future_shift(log_m, k)
        out[:, i] = shifted - log_m

    cols = [f"log_return_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


# =========================
# Classification labels
# =========================
def make_classification_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.5,
    use_smoothing: bool = True,
    dtype=np.float64,
) -> pd.DataFrame:

    if horizons is None:
        horizons = DEFAULT_HORIZONS

    m = _mid_price_array(df, dtype)
    n = len(m)

    out = np.empty((n, len(horizons)), dtype=dtype)

    for i, k in enumerate(horizons):
        if use_smoothing:
            future_m = _smoothed_mid(m, k)
        else:
            future_m = _future_shift(m, k)

        pct_change = (future_m - m) / m

        valid = pct_change[~np.isnan(pct_change)]
        sigma = np.std(valid)
        threshold = alpha * sigma

        labels = np.full(n, np.nan, dtype=dtype)

        up_mask = pct_change > threshold
        down_mask = pct_change < -threshold
        mid_mask = (~np.isnan(pct_change)) & (~up_mask) & (~down_mask)

        labels[up_mask] = 2
        labels[down_mask] = 0
        labels[mid_mask] = 1

        out[:, i] = labels

    cols = [f"label_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


def make_fixed_threshold_classification_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.002,
    use_smoothing: bool = True,
    dtype=np.float64,
) -> pd.DataFrame:

    if horizons is None:
        horizons = DEFAULT_HORIZONS

    m = _mid_price_array(df, dtype)
    n = len(m)

    out = np.empty((n, len(horizons)), dtype=dtype)

    for i, k in enumerate(horizons):
        if use_smoothing:
            future_m = _smoothed_mid(m, k)
        else:
            future_m = _future_shift(m, k)

        pct_change = (future_m - m) / m

        labels = np.full(n, np.nan, dtype=dtype)

        up_mask = pct_change > alpha
        down_mask = pct_change < -alpha
        mid_mask = (~np.isnan(pct_change)) & (~up_mask) & (~down_mask)

        labels[up_mask] = 2
        labels[down_mask] = 0
        labels[mid_mask] = 1

        out[:, i] = labels

    cols = [f"label_{k}" for k in horizons]
    return pd.DataFrame(out, index=df.index, columns=cols)


# =========================
# Combined pipeline
# =========================
def make_all_labels(
    df: pd.DataFrame,
    horizons: List[int] | None = None,
    alpha: float = 0.5,
    dtype=np.float64,
) -> pd.DataFrame:

    reg = make_regression_labels(df, horizons, dtype)
    cls = make_classification_labels(df, horizons, alpha=alpha, dtype=dtype)

    return pd.concat([reg, cls], axis=1)


# =========================
# Class weights
# =========================
def get_class_weights(labels: np.ndarray) -> np.ndarray:
    valid = labels[~np.isnan(labels)].astype(int)

    counts = np.bincount(valid, minlength=3).astype(float)
    counts = np.maximum(counts, 1.0)

    weights = len(valid) / (3.0 * counts)
    return weights

In [ ]:
import subprocess, sys

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    print('🚀 Colab detected — installing cuML for GPU acceleration...')
    print('   (This takes ~2-3 minutes, but training will be 10-50x faster)')

    # Install RAPIDS cuML (GPU-accelerated ML)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'cudf-cu12', 'cuml-cu12',
        '--extra-index-url=https://pypi.nvidia.com'
    ])
    print('✅ cuML installed successfully!')
else:
    print('💻 Local machine detected — skipping cuML install (will use CPU sklearn)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Imports & Setup
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, glob, gc, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (14, 6), 'font.size': 11})
sns.set_style('whitegrid')


USE_GPU = False
try:
    import cuml
    from cuml.ensemble import RandomForestClassifier as cuRFC
    from cuml.ensemble import RandomForestRegressor as cuRFR
    USE_GPU = True
    print('✓ cuML detected — using GPU-accelerated Random Forest')
except ImportError:
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
    print('✗ cuML not found — using CPU sklearn (install cuML for 10-50x speedup on Colab GPU)')

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from google.colab import drive
drive.mount('/content/drive')


DATA_DIR       = os.path.join(PROJECT_ROOT, 'drive', 'MyDrive', 'multi-horizon-ofi', 'data', 'processed')
WEIGHTS_DIR    = os.path.join(PROJECT_ROOT, 'drive', 'MyDrive', 'multi-horizon-ofi', 'model_weights')
RESULTS_DIR    = os.path.join(PROJECT_ROOT, 'drive', 'MyDrive', 'multi-horizon-ofi', 'results')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# # ── Feature engineering functions ─────────────────────────────────────────────
# from src.features.ofi import compute_multi_level_ofi
# from src.features.microstructure import compute_all_features
# from src.features.labels import (
#     make_regression_labels, make_classification_labels, DEFAULT_HORIZONS
# )

HORIZONS = DEFAULT_HORIZONS  # [10, 20, 50, 100]
# HORIZONS = [10, 20, 50, 100]

# ── Discover tickers ──────────────────────────────────────────────────────────
tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and glob.glob(os.path.join(DATA_DIR, d, '*.parquet'))
])

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Tickers      : {tickers}')
print(f'Horizons     : {HORIZONS}')
print(f'Weights dir  : {WEIGHTS_DIR}')
print(f'USE_GPU      : {USE_GPU}')

In [ ]:
"""
Train a Random Forest baseline one day at a time to avoid OOM.

Design:
- Load one parquet day at a time.
- Generate labels and windows for that day only.
- Train incrementally with warm_start (small tree batches per day).
- Free day arrays immediately after each file.
- Evaluate in a second pass over held-out files.
"""

import gc
import json
import logging
import os
import time
from contextlib import contextmanager

import joblib
import numpy as np
import pandas as pd
import psutil
from sklearn.ensemble import RandomForestClassifier


CONFIG = {
    "data_dir": str(DATA_DIR),
    "weights_dir": str(WEIGHTS_DIR),
    "results_dir": str(RESULTS_DIR),
    "ticker": None,
    "tickers": None,
    "seq_len": 100,
    "alpha": 0.00005,

    # Total trees per horizon in the final model.
    "n_estimators": 240,

    # Trees added per training day file (warm_start incremental fit).
    "trees_per_day": 12,

    "max_depth": 20,
    "min_samples_leaf": 20,

    # Keep low for RAM stability.
    "n_jobs": 1,

    "random_state": 42,

    # One-day-at-a-time controls.
    "train_file_fraction": 0.8,
    "max_files_per_ticker": 5,

    # Per-day sample caps (after subsampling/filtering).
    "max_rows_per_day_train": 12000,
    "max_rows_per_day_eval": 16000,

    # Adaptive subsampling based on free RAM.
    "min_free_ram_gb": 2.0,
    "base_subsample": 8,
    "high_pressure_subsample": 16,
    "critical_pressure_subsample": 32,

    # Logging / diagnostics.
    "log_level": "INFO",
    "warn_if_avail_below_gb": 2.0,
    "warn_if_proc_tree_above_gb": 12.0,

    # If True, raises when a day/horizon has only one class during training.
    # For robustness on sparse days, False is safer.
    "fail_if_single_class_split": False,
}


RAW_LOB_10_COLS = [
    f"{side}_{field}_{lvl}"
    for lvl in range(1, 11)
    for side, field in (
        ("ask", "price"),
        ("ask", "size"),
        ("bid", "price"),
        ("bid", "size"),
    )
]


# =========================
# Logger
# =========================
def _setup_logger(level: str = "INFO") -> logging.Logger:
    logger = logging.getLogger("rf_day_stream")

    if logger.handlers:
        return logger

    logger.setLevel(getattr(logging, level.upper(), logging.INFO))

    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))

    logger.addHandler(handler)
    logger.propagate = False
    return logger


LOGGER = _setup_logger(CONFIG.get("log_level", "INFO"))


# =========================
# Memory utilities
# =========================
def _mem_gb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / 1e9


def _avail_gb() -> float:
    return psutil.virtual_memory().available / 1e9


def _proc_tree_rss_gb() -> tuple[float, float, float]:
    p = psutil.Process(os.getpid())

    parent = p.memory_info().rss
    children = 0

    for c in p.children(recursive=True):
        try:
            children += c.memory_info().rss
        except Exception:
            pass

    parent_gb = parent / 1e9
    child_gb = children / 1e9
    total_gb = (parent + children) / 1e9
    return parent_gb, child_gb, total_gb


def _array_nbytes(arr: np.ndarray | None) -> int:
    if arr is None:
        return 0
    return int(getattr(arr, "nbytes", 0))


def _fmt_gb(nbytes: int) -> str:
    return f"{nbytes / 1e9:.3f} GB"


# =========================
# Logging helpers
# =========================
def _log_mem(tag: str, arrays: dict | None = None) -> None:
    parent_gb, child_gb, total_tree_gb = _proc_tree_rss_gb()
    avail = _avail_gb()
    vm = psutil.virtual_memory()

    msg = (
        f"[{tag}] parent={parent_gb:.2f}GB | children={child_gb:.2f}GB "
        f"| total={total_tree_gb:.2f}GB | avail={avail:.2f}GB | used={vm.percent:.1f}%"
    )

    if arrays:
        parts = []
        for name, arr in arrays.items():
            if arr is None:
                parts.append(f"{name}=None")
            else:
                parts.append(f"{name}.shape={arr.shape}, {name}.bytes={_fmt_gb(_array_nbytes(arr))}")
        msg += " | " + " ; ".join(parts)

    LOGGER.info(msg)

    if avail < CONFIG["warn_if_avail_below_gb"]:
        LOGGER.warning(f"[{tag}] LOW RAM: {avail:.2f} GB")

    if total_tree_gb > CONFIG["warn_if_proc_tree_above_gb"]:
        LOGGER.warning(f"[{tag}] HIGH PROCESS RAM: {total_tree_gb:.2f} GB")


@contextmanager
def _stage(name: str):
    t0 = time.time()
    _log_mem(f"{name} START")
    try:
        yield
    finally:
        _log_mem(f"{name} END")
        LOGGER.info(f"[{name}] time={time.time() - t0:.2f}s")


# =========================
# File discovery and split
# =========================
def _discover_tickers(data_dir: str) -> list[str]:
    tickers = []
    for d in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, d)
        if not os.path.isdir(p):
            continue

        for f in os.listdir(p):
            if f.endswith(".parquet"):
                tickers.append(d)
                break
    return tickers


def resolve_tickers(cfg: dict) -> list[str]:
    if cfg["tickers"]:
        return cfg["tickers"]
    if cfg["ticker"]:
        return [cfg["ticker"]]
    return _discover_tickers(cfg["data_dir"])


def _collect_files_by_ticker(data_dir: str, tickers: list[str], max_files_per_ticker: int) -> dict[str, list[str]]:
    out: dict[str, list[str]] = {}

    for ticker in tickers:
        t_dir = os.path.join(data_dir, ticker)
        if not os.path.isdir(t_dir):
            continue

        files = sorted(
            os.path.join(t_dir, f)
            for f in os.listdir(t_dir)
            if f.endswith(".parquet")
        )

        if max_files_per_ticker > 0:
            files = files[:max_files_per_ticker]

        if files:
            out[ticker] = files

    return out


def _split_train_eval_files(files_by_ticker: dict[str, list[str]], train_frac: float) -> tuple[list[tuple[str, str]], list[tuple[str, str]]]:
    train_files: list[tuple[str, str]] = []
    eval_files: list[tuple[str, str]] = []

    for ticker, files in files_by_ticker.items():
        n = len(files)

        if n == 1:
            n_train = 1
        else:
            n_train = int(np.floor(n * train_frac))
            n_train = max(1, min(n - 1, n_train))

        for i, p in enumerate(files):
            if i < n_train:
                train_files.append((ticker, p))
            else:
                eval_files.append((ticker, p))

    return train_files, eval_files


# =========================
# Sampling controls
# =========================
def _choose_subsample(cfg: dict) -> int:
    free = _avail_gb()

    if free < max(1.0, cfg["min_free_ram_gb"] * 0.5):
        return max(cfg["critical_pressure_subsample"], cfg["base_subsample"])

    if free < cfg["min_free_ram_gb"]:
        return max(cfg["high_pressure_subsample"], cfg["base_subsample"])

    return cfg["base_subsample"]


def _choose_max_rows_for_day(cfg: dict, is_train: bool) -> int:
    base = int(cfg["max_rows_per_day_train"] if is_train else cfg["max_rows_per_day_eval"])
    free = _avail_gb()

    if free < max(1.0, cfg["min_free_ram_gb"] * 0.5):
        return max(2000, base // 4)

    if free < cfg["min_free_ram_gb"]:
        return max(4000, base // 2)

    return base


# =========================
# Day sample builder
# =========================
def _build_sampled_windows(
    raw: np.ndarray,
    seq_len: int,
    starts: np.ndarray,
    chunk_size: int = 2048,
) -> np.ndarray:
    """
    Build flattened windows only for requested start indices.
    This avoids materializing all windows for a day.
    """
    n_samples = int(starts.size)
    feat_dim = seq_len * raw.shape[1]

    out = np.empty((n_samples, feat_dim), dtype=np.float32)
    offsets = np.arange(seq_len, dtype=np.int64)

    write = 0
    while write < n_samples:
        end = min(write + chunk_size, n_samples)
        s = starts[write:end]

        block = raw[s[:, None] + offsets[None, :], :]
        out[write:end] = block.reshape(len(s), feat_dim)

        write = end

    return out


def _build_day_samples(
    parquet_path: str,
    horizons: list[int],
    seq_len: int,
    alpha: float,
    step: int,
    max_rows: int,
) -> tuple[np.ndarray, np.ndarray]:
    with _stage(f"build_day {os.path.basename(parquet_path)}"):
        df = pd.read_parquet(parquet_path, columns=RAW_LOB_10_COLS)
        raw = df.to_numpy(dtype=np.float32, copy=False)

        _log_mem("after read_parquet", {"raw": raw})

        # Keep float labels first so NaN filtering works correctly.
        y_float = make_fixed_threshold_classification_labels(
            df,
            horizons=horizons,
            alpha=alpha,
            use_smoothing=True,
        ).to_numpy(dtype=np.float32, copy=False)

        _log_mem("after label generation", {"y_float": y_float})

        max_h = max(horizons)
        valid_end = len(df) - max_h

        if valid_end <= seq_len:
            del df, raw, y_float
            gc.collect()
            return (
                np.empty((0, seq_len * len(RAW_LOB_10_COLS)), dtype=np.float32),
                np.empty((0, len(horizons)), dtype=np.int8),
            )

        # labels aligned to window starts [0, valid_count)
        labels = y_float[seq_len - 1 : valid_end]
        valid_count = labels.shape[0]

        if valid_count <= 0:
            del df, raw, y_float, labels
            gc.collect()
            return (
                np.empty((0, seq_len * len(RAW_LOB_10_COLS)), dtype=np.float32),
                np.empty((0, len(horizons)), dtype=np.int8),
            )

        # Keep only rows where all horizons have non-NaN labels.
        valid_mask = ~np.isnan(labels).any(axis=1)
        starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

        if step > 1:
            starts = starts[::step]

        if max_rows > 0:
            starts = starts[:max_rows]

        if starts.size == 0:
            del df, raw, y_float, labels, valid_mask, starts
            gc.collect()
            return (
                np.empty((0, seq_len * len(RAW_LOB_10_COLS)), dtype=np.float32),
                np.empty((0, len(horizons)), dtype=np.int8),
            )

        x_day = _build_sampled_windows(raw, seq_len, starts)
        y_day = labels[starts].astype(np.int8, copy=False)

        # Keep only valid class labels in [0, 1, 2] for all horizons.
        class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
        x_day = x_day[class_mask]
        y_day = y_day[class_mask]

        _log_mem("after day build", {"x_day": x_day, "y_day": y_day})

        del df, raw, y_float, labels, valid_mask, starts, class_mask
        gc.collect()

        _log_mem("after day cleanup")
        return x_day, y_day


# =========================
# Metrics helpers
# =========================
def _update_confusion_matrix(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    y_true = y_true.astype(np.int64, copy=False)
    y_pred = y_pred.astype(np.int64, copy=False)

    idx = y_true * 3 + y_pred
    cm += np.bincount(idx, minlength=9).reshape(3, 3)


def _metrics_from_confusion(cm: np.ndarray, h: int) -> dict:
    cm = cm.astype(np.float64, copy=False)

    tp = np.diag(cm)
    support = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)
    total = support.sum()

    if total == 0:
        return {
            f"h{h}_accuracy": 0.0,
            f"h{h}_f1_macro": 0.0,
            f"h{h}_f1_weighted": 0.0,
            f"h{h}_precision_macro": 0.0,
            f"h{h}_recall_macro": 0.0,
        }

    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)

    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    accuracy = float(tp.sum() / total)
    f1_macro = float(np.mean(f1))
    f1_weighted = float(np.sum(f1 * support) / total)
    precision_macro = float(np.mean(precision))
    recall_macro = float(np.mean(recall))

    return {
        f"h{h}_accuracy": accuracy,
        f"h{h}_f1_macro": f1_macro,
        f"h{h}_f1_weighted": f1_weighted,
        f"h{h}_precision_macro": precision_macro,
        f"h{h}_recall_macro": recall_macro,
    }


def _class_count_str(y: np.ndarray, n_classes: int = 3) -> str:
    counts = np.bincount(y.astype(np.int64, copy=False), minlength=n_classes)
    return " ".join(f"c{i}={int(c)}" for i, c in enumerate(counts))


def _make_rf(cfg: dict, n_estimators: int) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=cfg["max_depth"],
        min_samples_leaf=cfg["min_samples_leaf"],
        class_weight="balanced",
        n_jobs=cfg["n_jobs"],
        random_state=cfg["random_state"],
        warm_start=True,
    )


# =========================
# Main day-by-day pipeline
# =========================
def main_day_by_day(config: dict | None = None) -> dict:
    cfg = dict(CONFIG)
    if config is not None:
        cfg.update(config)

    if "make_fixed_threshold_classification_labels" not in globals():
        raise RuntimeError(
            "make_fixed_threshold_classification_labels is not defined. "
            "Run the labels cell before this training cell."
        )

    if "DEFAULT_HORIZONS" in globals():
        horizons = list(DEFAULT_HORIZONS)
    else:
        horizons = [10, 20, 50, 100]

    os.makedirs(cfg["weights_dir"], exist_ok=True)
    os.makedirs(cfg["results_dir"], exist_ok=True)

    tickers = resolve_tickers(cfg)
    if not tickers:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _collect_files_by_ticker(
        cfg["data_dir"],
        tickers,
        int(cfg["max_files_per_ticker"]),
    )

    if not files_by_ticker:
        raise FileNotFoundError("No parquet files found for requested tickers.")

    train_files, eval_files = _split_train_eval_files(
        files_by_ticker,
        float(cfg["train_file_fraction"]),
    )

    LOGGER.info("=" * 80)
    LOGGER.info("RF pipeline (day-by-day streaming)")
    LOGGER.info("=" * 80)
    LOGGER.info("Tickers=%s | Horizons=%s", tickers, horizons)
    LOGGER.info("Seq len=%d | Alpha=%.6f", cfg["seq_len"], cfg["alpha"])
    LOGGER.info(
        "Files: train=%d eval=%d | max_files_per_ticker=%d",
        len(train_files),
        len(eval_files),
        cfg["max_files_per_ticker"],
    )
    LOGGER.info(
        "Per-day limits: train=%d eval=%d | base_subsample=%d",
        cfg["max_rows_per_day_train"],
        cfg["max_rows_per_day_eval"],
        cfg["base_subsample"],
    )

    _log_mem("pipeline start")
    t0 = time.time()

    models: dict[int, RandomForestClassifier | None] = {h: None for h in horizons}
    fitted_trees: dict[int, int] = {h: 0 for h in horizons}
    fitted_days: dict[int, int] = {h: 0 for h in horizons}
    train_rows_seen: dict[int, int] = {h: 0 for h in horizons}

    confusion: dict[int, np.ndarray] = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    eval_rows_seen: dict[int, int] = {h: 0 for h in horizons}

    # -------------------------
    # Pass 1: training files
    # -------------------------
    for file_idx, (ticker, path) in enumerate(train_files, start=1):
        sub = _choose_subsample(cfg)
        max_rows = _choose_max_rows_for_day(cfg, is_train=True)

        LOGGER.info(
            "[train %d/%d] ticker=%s file=%s | sub=%d max_rows=%d",
            file_idx,
            len(train_files),
            ticker,
            os.path.basename(path),
            sub,
            max_rows,
        )

        x_day, y_day = _build_day_samples(
            parquet_path=path,
            horizons=horizons,
            seq_len=cfg["seq_len"],
            alpha=cfg["alpha"],
            step=sub,
            max_rows=max_rows,
        )

        if x_day.shape[0] == 0:
            LOGGER.warning("Skipping empty day: %s", path)
            del x_day, y_day
            gc.collect()
            continue

        for i, h in enumerate(horizons):
            y_h = y_day[:, i]

            valid_mask = (y_h >= 0) & (y_h <= 2)
            if not np.any(valid_mask):
                LOGGER.warning("[h=%d] no valid labels for %s", h, os.path.basename(path))
                continue

            if np.all(valid_mask):
                x_fit = x_day
                y_fit = y_h
            else:
                x_fit = x_day[valid_mask]
                y_fit = y_h[valid_mask]

            unique_classes = int(np.unique(y_fit).size)
            if unique_classes < 2:
                msg = f"[h={h}] single-class day for {os.path.basename(path)}"
                if cfg.get("fail_if_single_class_split", False):
                    raise ValueError(msg)
                LOGGER.warning(msg + " -> skipped")
                if x_fit is not x_day:
                    del x_fit
                del y_fit
                continue

            remaining = int(cfg["n_estimators"]) - fitted_trees[h]
            if remaining <= 0:
                if x_fit is not x_day:
                    del x_fit
                del y_fit
                continue

            add_trees = min(int(cfg["trees_per_day"]), remaining)

            if models[h] is None:
                models[h] = _make_rf(cfg, add_trees)
            else:
                models[h].set_params(n_estimators=models[h].n_estimators + add_trees)

            fit_t0 = time.time()
            models[h].fit(x_fit, y_fit)
            fit_s = time.time() - fit_t0

            fitted_trees[h] = int(models[h].n_estimators)
            fitted_days[h] += 1
            train_rows_seen[h] += int(y_fit.size)

            LOGGER.info(
                "[h=%d] day_fit rows=%d classes=%s trees=%d/%d fit_time=%.2fs",
                h,
                int(y_fit.size),
                _class_count_str(y_fit),
                fitted_trees[h],
                int(cfg["n_estimators"]),
                fit_s,
            )

            if x_fit is not x_day:
                del x_fit
            del y_fit

        del x_day, y_day
        gc.collect()
        _log_mem(f"after train file {os.path.basename(path)}")

    # -------------------------
    # Pass 2: evaluation files
    # -------------------------
    for file_idx, (ticker, path) in enumerate(eval_files, start=1):
        sub = _choose_subsample(cfg)
        max_rows = _choose_max_rows_for_day(cfg, is_train=False)

        LOGGER.info(
            "[eval %d/%d] ticker=%s file=%s | sub=%d max_rows=%d",
            file_idx,
            len(eval_files),
            ticker,
            os.path.basename(path),
            sub,
            max_rows,
        )

        x_day, y_day = _build_day_samples(
            parquet_path=path,
            horizons=horizons,
            seq_len=cfg["seq_len"],
            alpha=cfg["alpha"],
            step=sub,
            max_rows=max_rows,
        )

        if x_day.shape[0] == 0:
            LOGGER.warning("Skipping empty eval day: %s", path)
            del x_day, y_day
            gc.collect()
            continue

        for i, h in enumerate(horizons):
            model = models[h]
            if model is None:
                continue

            y_h = y_day[:, i]
            valid_mask = (y_h >= 0) & (y_h <= 2)
            if not np.any(valid_mask):
                continue

            if np.all(valid_mask):
                x_eval = x_day
                y_eval = y_h
            else:
                x_eval = x_day[valid_mask]
                y_eval = y_h[valid_mask]

            pred_t0 = time.time()
            y_pred = model.predict(x_eval)
            pred_s = time.time() - pred_t0

            _update_confusion_matrix(confusion[h], y_eval, y_pred)
            eval_rows_seen[h] += int(y_eval.size)

            LOGGER.info(
                "[h=%d] day_eval rows=%d pred_time=%.2fs",
                h,
                int(y_eval.size),
                pred_s,
            )

            del y_pred
            if x_eval is not x_day:
                del x_eval
            del y_eval

        del x_day, y_day
        gc.collect()
        _log_mem(f"after eval file {os.path.basename(path)}")

    # -------------------------
    # Save models + compute metrics
    # -------------------------
    metrics: dict[str, float] = {}
    model_paths: dict[str, str] = {}

    for h in horizons:
        metrics.update(_metrics_from_confusion(confusion[h], h))

        if models[h] is None:
            LOGGER.warning("[h=%d] model was never fitted; skipping save", h)
            continue

        out_path = os.path.join(cfg["weights_dir"], f"rf_wallbridge_classifier_h{h}.joblib")
        joblib.dump(models[h], out_path)
        model_paths[f"h{h}"] = out_path

        LOGGER.info(
            "[h=%d] saved model trees=%d train_rows=%d eval_rows=%d",
            h,
            fitted_trees[h],
            train_rows_seen[h],
            eval_rows_seen[h],
        )

    total_s = time.time() - t0

    run_meta = {
        "timestamp": pd.Timestamp.now().isoformat(),
        "mode": "day_by_day_streaming",
        "tickers": tickers,
        "horizons": horizons,
        "seq_len": int(cfg["seq_len"]),
        "alpha": float(cfg["alpha"]),
        "rf_params": {
            "n_estimators_target": int(cfg["n_estimators"]),
            "trees_per_day": int(cfg["trees_per_day"]),
            "max_depth": int(cfg["max_depth"]),
            "min_samples_leaf": int(cfg["min_samples_leaf"]),
            "n_jobs": int(cfg["n_jobs"]),
        },
        "sampling": {
            "train_file_fraction": float(cfg["train_file_fraction"]),
            "base_subsample": int(cfg["base_subsample"]),
            "high_pressure_subsample": int(cfg["high_pressure_subsample"]),
            "critical_pressure_subsample": int(cfg["critical_pressure_subsample"]),
            "max_rows_per_day_train": int(cfg["max_rows_per_day_train"]),
            "max_rows_per_day_eval": int(cfg["max_rows_per_day_eval"]),
        },
        "files": {
            "train_files": len(train_files),
            "eval_files": len(eval_files),
        },
        "fitted_trees": {f"h{h}": fitted_trees[h] for h in horizons},
        "fitted_days": {f"h{h}": fitted_days[h] for h in horizons},
        "train_rows_seen": {f"h{h}": train_rows_seen[h] for h in horizons},
        "eval_rows_seen": {f"h{h}": eval_rows_seen[h] for h in horizons},
        "test_metrics": metrics,
        "model_paths": model_paths,
        "runtime_seconds": round(total_s, 2),
    }

    results_path = os.path.join(cfg["results_dir"], "rf_wallbridge_results.json")
    with open(results_path, "w") as f:
        json.dump(run_meta, f, indent=2)

    LOGGER.info("=" * 80)
    LOGGER.info("Done | results=%s", results_path)
    _log_mem("pipeline end")

    return run_meta


run_meta = main_day_by_day()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Deep Learning Streaming Setup (Memory-safe)
# ══════════════════════════════════════════════════════════════════════════════
import os
import gc
import json
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import psutil
except ImportError:
    psutil = None


DEEP_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEEP_DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


DEEP_RAW_LOB_10_COLS = globals().get(
    'RAW_LOB_10_COLS',
    [
        f"{side}_{field}_{lvl}"
        for lvl in range(1, 11)
        for side, field in (
            ('ask', 'price'),
            ('ask', 'size'),
            ('bid', 'price'),
            ('bid', 'size'),
        )
    ],
)

DEEP_HORIZONS = list(globals().get('DEFAULT_HORIZONS', [10, 20, 50, 100]))

DEEP_CONFIG = {
    'data_dir': str(globals().get('DATA_DIR', os.path.join(os.getcwd(), 'data', 'processed'))),
    'weights_dir': str(globals().get('WEIGHTS_DIR', os.path.join(os.getcwd(), 'model_weights'))),
    'results_dir': str(globals().get('RESULTS_DIR', os.path.join(os.getcwd(), 'results'))),
    'ticker': None,
    'tickers': None,
    'horizons': DEEP_HORIZONS,
    'seq_len': 100,
    'alpha': float(globals().get('CONFIG', {}).get('alpha', 0.00005) if 'CONFIG' in globals() else 0.00005),
    'train_file_fraction': 0.8,
    'max_files_per_ticker': 5,
    'base_subsample': 8,
    'high_pressure_subsample': 16,
    'critical_pressure_subsample': 32,
    'max_rows_per_day_train': 12000,
    'max_rows_per_day_eval': 16000,
    'min_free_ram_gb': 2.0,
    'batch_size': 64,
    'epochs': 1,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'num_workers': 0,
    'amp': True,
    'seed': 42,
    'run_architectures': [
        'dilated_transformer',
        'hybrid_cnn_inception_lstm',
        'seq2seq_attn',
    ],
}


@dataclass
class DayStats:
    ticker: str
    file_name: str
    rows_raw: int
    rows_kept: int
    step: int
    max_rows: int


class DayWindowDataset(Dataset):
    """Memory-efficient dataset that materializes windows per batch, not upfront."""

    def __init__(self, raw: np.ndarray, starts: np.ndarray, labels: np.ndarray, seq_len: int):
        self.raw = raw
        self.starts = starts.astype(np.int64, copy=False)
        self.labels = labels.astype(np.int64, copy=False)
        self.seq_len = int(seq_len)

    def __len__(self) -> int:
        return int(self.starts.size)

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        x = self.raw[s : s + self.seq_len]
        y = self.labels[idx]
        return torch.from_numpy(x), torch.from_numpy(y)


# -----------------------------
# Repro + memory helpers
# -----------------------------
def deep_set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def deep_avail_gb() -> float:
    if psutil is None:
        return 999.0
    return psutil.virtual_memory().available / 1e9


def deep_proc_gb() -> float:
    if psutil is None:
        return 0.0
    return psutil.Process(os.getpid()).memory_info().rss / 1e9


def deep_mem(tag: str) -> None:
    avail = deep_avail_gb()
    proc = deep_proc_gb()
    print(f"[{tag}] proc={proc:.2f}GB avail={avail:.2f}GB")


# -----------------------------
# File discovery/splitting
# -----------------------------
def _deep_discover_tickers(data_dir: str) -> List[str]:
    tickers: List[str] = []
    if not os.path.isdir(data_dir):
        return tickers

    for d in sorted(os.listdir(data_dir)):
        p = os.path.join(data_dir, d)
        if not os.path.isdir(p):
            continue
        has_parquet = any(f.endswith('.parquet') for f in os.listdir(p))
        if has_parquet:
            tickers.append(d)
    return tickers


def _deep_resolve_tickers(cfg: dict) -> List[str]:
    if cfg.get('tickers'):
        return list(cfg['tickers'])
    if cfg.get('ticker'):
        return [cfg['ticker']]
    return _deep_discover_tickers(cfg['data_dir'])


def _deep_collect_files_by_ticker(data_dir: str, tickers: List[str], max_files_per_ticker: int) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for ticker in tickers:
        t_dir = os.path.join(data_dir, ticker)
        if not os.path.isdir(t_dir):
            continue

        files = sorted(
            os.path.join(t_dir, f)
            for f in os.listdir(t_dir)
            if f.endswith('.parquet')
        )

        if max_files_per_ticker > 0:
            files = files[:max_files_per_ticker]

        if files:
            out[ticker] = files
    return out


def _deep_split_train_eval_files(files_by_ticker: Dict[str, List[str]], train_frac: float) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]]]:
    train_files: List[Tuple[str, str]] = []
    eval_files: List[Tuple[str, str]] = []

    for ticker, files in files_by_ticker.items():
        n = len(files)
        if n <= 1:
            n_train = 1
        else:
            n_train = int(np.floor(n * train_frac))
            n_train = max(1, min(n - 1, n_train))

        for i, p in enumerate(files):
            if i < n_train:
                train_files.append((ticker, p))
            else:
                eval_files.append((ticker, p))

    return train_files, eval_files


# -----------------------------
# Day-level subsampling rules
# -----------------------------
def _deep_choose_subsample(cfg: dict) -> int:
    free = deep_avail_gb()
    if free < max(1.0, cfg['min_free_ram_gb'] * 0.5):
        return max(cfg['critical_pressure_subsample'], cfg['base_subsample'])
    if free < cfg['min_free_ram_gb']:
        return max(cfg['high_pressure_subsample'], cfg['base_subsample'])
    return cfg['base_subsample']


def _deep_choose_max_rows(cfg: dict, is_train: bool) -> int:
    base = int(cfg['max_rows_per_day_train'] if is_train else cfg['max_rows_per_day_eval'])
    free = deep_avail_gb()

    if free < max(1.0, cfg['min_free_ram_gb'] * 0.5):
        return max(2000, base // 4)
    if free < cfg['min_free_ram_gb']:
        return max(4000, base // 2)
    return base


# -----------------------------
# Day dataset builder
# -----------------------------
def _deep_build_day_dataset(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[DayWindowDataset | None, DayStats | None]:
    if 'make_fixed_threshold_classification_labels' not in globals():
        raise RuntimeError(
            'make_fixed_threshold_classification_labels is not defined. Run labels cell first.'
        )

    horizons = list(cfg['horizons'])
    seq_len = int(cfg['seq_len'])
    alpha = float(cfg['alpha'])

    step = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    raw = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))

    y_float = make_fixed_threshold_classification_labels(
        df,
        horizons=horizons,
        alpha=alpha,
        use_smoothing=True,
    ).to_numpy(dtype=np.float32, copy=False)

    max_h = int(max(horizons))
    valid_end = len(df) - max_h

    if valid_end <= seq_len:
        del df, raw, y_float
        gc.collect()
        return None, None

    labels = y_float[seq_len - 1 : valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

    if step > 1:
        starts = starts[::step]

    if max_rows > 0:
        starts = starts[:max_rows]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts
        gc.collect()
        return None, None

    y_day = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts = starts[class_mask]
    y_day = y_day[class_mask]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
        gc.collect()
        return None, None

    ds = DayWindowDataset(raw=raw, starts=starts, labels=y_day, seq_len=seq_len)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_raw=int(len(df)),
        rows_kept=int(starts.size),
        step=int(step),
        max_rows=int(max_rows),
    )

    del df, y_float, labels, valid_mask, starts, y_day, class_mask
    gc.collect()
    return ds, stats


def _deep_make_loader(dataset: DayWindowDataset, cfg: dict, is_train: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=int(cfg['batch_size']),
        shuffle=bool(is_train),
        num_workers=int(cfg['num_workers']),
        pin_memory=(DEEP_DEVICE.type == 'cuda'),
        drop_last=False,
    )


def _deep_update_confusion(cm: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    idx = y_true.astype(np.int64, copy=False) * 3 + y_pred.astype(np.int64, copy=False)
    cm += np.bincount(idx, minlength=9).reshape(3, 3)


def _deep_metrics_from_confusion(cm: np.ndarray, h: int) -> Dict[str, float]:
    cm = cm.astype(np.float64, copy=False)
    tp = np.diag(cm)
    support = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)
    total = support.sum()

    if total == 0:
        return {
            f'h{h}_accuracy': 0.0,
            f'h{h}_f1_macro': 0.0,
            f'h{h}_f1_weighted': 0.0,
            f'h{h}_precision_macro': 0.0,
            f'h{h}_recall_macro': 0.0,
        }

    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(
        2.0 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    return {
        f'h{h}_accuracy': float(tp.sum() / total),
        f'h{h}_f1_macro': float(np.mean(f1)),
        f'h{h}_f1_weighted': float(np.sum(f1 * support) / total),
        f'h{h}_precision_macro': float(np.mean(precision)),
        f'h{h}_recall_macro': float(np.mean(recall)),
    }


def _deep_class_weights(labels: np.ndarray) -> List[np.ndarray]:
    weights: List[np.ndarray] = []
    for i in range(labels.shape[1]):
        y = labels[:, i]
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64)
        counts = np.maximum(counts, 1.0)
        w = counts.sum() / (3.0 * counts)
        w = np.clip(w, 0.25, 8.0)
        weights.append(w.astype(np.float32))
    return weights


def _deep_multihorizon_loss(logits: torch.Tensor, targets: torch.Tensor, weight_tensors: List[torch.Tensor] | None = None) -> torch.Tensor:
    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        losses.append(F.cross_entropy(logits[:, i, :], targets[:, i], weight=w))
    return torch.stack(losses).mean()


def _deep_cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"Deep training device: {DEEP_DEVICE}")
print(f"Deep data dir: {DEEP_CONFIG['data_dir']}")
print(f"Horizons: {DEEP_CONFIG['horizons']}")
print('Cell 7 ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Deep Model Definitions
# 1) Dilated CNN + Masked Multihead Attention (Transformer-style)
# 2) Hybrid CNN + Inception + LSTM
# 3) Seq2Seq + Attention (DeepLOB encoder + LSTM decoder)
# ══════════════════════════════════════════════════════════════════════════════

class CausalDilatedConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation, padding=self.pad)
        self.bn = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.res = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.conv(x)
        if self.pad > 0:
            y = y[..., :-self.pad]
        y = self.drop(F.gelu(self.bn(y)))
        r = self.res(x)
        if r.shape[-1] != y.shape[-1]:
            r = r[..., -y.shape[-1]:]
        return y + r


class DilatedMaskedTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        d_model: int = 96,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.15,
        max_len: int = 1024,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.input_proj = nn.Conv1d(input_dim, d_model, kernel_size=1)
        self.tcn = nn.ModuleList(
            [
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=1, dropout=dropout),
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=2, dropout=dropout),
                CausalDilatedConv1d(d_model, d_model, kernel_size=3, dilation=4, dropout=dropout),
            ]
        )

        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, _ = x.shape

        z = x.transpose(1, 2)
        z = self.input_proj(z)
        for block in self.tcn:
            z = block(z)
        z = z.transpose(1, 2)

        if self.pos_emb.size(1) >= seq_len:
            pos = self.pos_emb[:, :seq_len]
        else:
            pos = F.interpolate(
                self.pos_emb.transpose(1, 2),
                size=seq_len,
                mode='linear',
                align_corners=False,
            ).transpose(1, 2)

        z = z + pos
        attn_mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=z.device),
            diagonal=1,
        )
        z = self.encoder(z, mask=attn_mask)
        z = self.norm(z[:, -1, :])

        out = self.head(z)
        out = out.view(bsz, self.horizon_count, self.num_classes)
        return out


class InceptionBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b1 = out_ch // 4
        b2 = out_ch // 4
        b3 = out_ch // 4
        b4 = out_ch - (b1 + b2 + b3)

        self.branch1 = nn.Sequential(
            nn.Conv1d(in_ch, b1, kernel_size=1),
            nn.BatchNorm1d(b1),
            nn.GELU(),
        )
        self.branch2 = nn.Sequential(
            nn.Conv1d(in_ch, b2, kernel_size=1),
            nn.BatchNorm1d(b2),
            nn.GELU(),
            nn.Conv1d(b2, b2, kernel_size=3, padding=1),
            nn.BatchNorm1d(b2),
            nn.GELU(),
        )
        self.branch3 = nn.Sequential(
            nn.Conv1d(in_ch, b3, kernel_size=1),
            nn.BatchNorm1d(b3),
            nn.GELU(),
            nn.Conv1d(b3, b3, kernel_size=5, padding=2),
            nn.BatchNorm1d(b3),
            nn.GELU(),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_ch, b4, kernel_size=1),
            nn.BatchNorm1d(b4),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x),
        ], dim=1)


class HybridCNNInceptionLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        channels: int = 96,
        lstm_hidden: int = 96,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.stem = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.GELU(),
        )
        self.inception = InceptionBlock1D(channels, channels)
        self.lstm = nn.LSTM(
            input_size=channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden, horizon_count * num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)
        z = x.transpose(1, 2)
        z = self.stem(z)
        z = self.inception(z)
        z = z.transpose(1, 2)
        z, _ = self.lstm(z)
        z = self.dropout(z[:, -1, :])

        out = self.head(z)
        out = out.view(bsz, self.horizon_count, self.num_classes)
        return out


class DeepLOBEncoder(nn.Module):
    def __init__(self, conv_channels: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(1, 2), stride=(1, 2)),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 32, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, conv_channels, kernel_size=(3, 1), padding=(1, 0)),
            nn.BatchNorm2d(conv_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, F]
        z = x.unsqueeze(1)
        z = self.conv(z)
        z = z.mean(dim=3)
        z = z.transpose(1, 2)
        return z


class Seq2SeqDeepLOBAttention(nn.Module):
    def __init__(
        self,
        input_dim: int,
        horizon_count: int,
        num_classes: int = 3,
        hidden_dim: int = 96,
        embed_dim: int = 16,
    ):
        super().__init__()
        self.horizon_count = horizon_count
        self.num_classes = num_classes

        self.encoder_cnn = DeepLOBEncoder(conv_channels=64)
        self.encoder_lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )

        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.decoder_cell = nn.LSTMCell(hidden_dim + embed_dim, hidden_dim)
        self.query_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def _attend(self, enc_out: torch.Tensor, h_t: torch.Tensor) -> torch.Tensor:
        # enc_out: [B, T, H], h_t: [B, H]
        q = self.query_proj(h_t).unsqueeze(2)
        score = torch.bmm(enc_out, q).squeeze(2)
        alpha = torch.softmax(score, dim=1)
        ctx = torch.bmm(alpha.unsqueeze(1), enc_out).squeeze(1)
        return ctx

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz = x.size(0)

        enc_in = self.encoder_cnn(x)
        enc_out, (h_n, c_n) = self.encoder_lstm(enc_in)

        h_t = h_n[-1]
        c_t = c_n[-1]

        prev_cls = torch.ones(bsz, dtype=torch.long, device=x.device)
        logits_steps = []

        for _ in range(self.horizon_count):
            cls_vec = self.class_embed(prev_cls)
            ctx = self._attend(enc_out, h_t)
            dec_in = torch.cat([ctx, cls_vec], dim=1)
            h_t, c_t = self.decoder_cell(dec_in, (h_t, c_t))

            step_logits = self.classifier(torch.cat([h_t, ctx], dim=1))
            logits_steps.append(step_logits)
            prev_cls = step_logits.argmax(dim=1)

        return torch.stack(logits_steps, dim=1)


def build_deep_model(arch: str, input_dim: int, horizon_count: int, num_classes: int = 3) -> nn.Module:
    if arch == 'dilated_transformer':
        return DilatedMaskedTransformer(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            d_model=96,
            n_heads=4,
            n_layers=2,
            dropout=0.15,
        )

    if arch == 'hybrid_cnn_inception_lstm':
        return HybridCNNInceptionLSTM(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            channels=96,
            lstm_hidden=96,
            dropout=0.2,
        )

    if arch == 'seq2seq_attn':
        return Seq2SeqDeepLOBAttention(
            input_dim=input_dim,
            horizon_count=horizon_count,
            num_classes=num_classes,
            hidden_dim=96,
            embed_dim=16,
        )

    raise ValueError(f'Unknown architecture: {arch}')


print('Cell 8 ready: deep model classes loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Train/Eval Runner (Day-by-day streaming) + Save Weights/Results
# ══════════════════════════════════════════════════════════════════════════════

def train_one_deep_architecture(arch: str, config: dict | None = None) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    horizons = list(cfg['horizons'])

    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    tickers = _deep_resolve_tickers(cfg)
    if not tickers:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg['data_dir'],
        tickers,
        int(cfg['max_files_per_ticker']),
    )
    if not files_by_ticker:
        raise FileNotFoundError('No parquet files found for selected tickers.')

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker,
        float(cfg['train_file_fraction']),
    )

    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
    )

    amp_enabled = bool(cfg.get('amp', True)) and DEEP_DEVICE.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    confusion = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    train_rows_seen = {h: 0 for h in horizons}
    eval_rows_seen = {h: 0 for h in horizons}
    train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    total_train_days = 0
    start_t = time.time()

    print('\n' + '=' * 88)
    print(f"Training architecture: {arch}")
    print(f"Device={DEEP_DEVICE} | train_files={len(train_files)} eval_files={len(eval_files)}")
    deep_mem(f"{arch} start")

    # -------------------------
    # Pass 1: Train day-by-day
    # -------------------------
    for epoch in range(int(cfg['epochs'])):
        model.train()
        print(f"\n[{arch}] epoch {epoch + 1}/{int(cfg['epochs'])}")

        for file_idx, (ticker, path) in enumerate(train_files, start=1):
            ds, stats = _deep_build_day_dataset(path, cfg, is_train=True)
            if ds is None:
                print(f"[{arch}][train {file_idx}/{len(train_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=True)
            day_weights_np = _deep_class_weights(ds.labels)
            day_weights = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in day_weights_np]

            loss_sum = 0.0
            n_batches = 0

            for xb, yb in loader:
                xb = xb.to(DEEP_DEVICE, non_blocking=True)
                yb = yb.to(DEEP_DEVICE, non_blocking=True, dtype=torch.long)

                optimizer.zero_grad(set_to_none=True)

                if amp_enabled:
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        logits = model(xb)
                        loss = _deep_multihorizon_loss(logits, yb, day_weights)

                    scaler.scale(loss).backward()
                    if float(cfg['grad_clip']) > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['grad_clip']))
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    logits = model(xb)
                    loss = _deep_multihorizon_loss(logits, yb, day_weights)
                    loss.backward()
                    if float(cfg['grad_clip']) > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['grad_clip']))
                    optimizer.step()

                loss_sum += float(loss.item())
                n_batches += 1

                del xb, yb, logits, loss

            for i, h in enumerate(horizons):
                train_rows_seen[h] += int(ds.labels.shape[0])
                train_class_counts[h] += np.bincount(ds.labels[:, i], minlength=3)

            total_train_days += 1
            day_loss = loss_sum / max(1, n_batches)
            print(
                f"[{arch}][train {file_idx}/{len(train_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows} loss={day_loss:.4f}"
            )

            del loader, ds, day_weights, day_weights_np
            _deep_cleanup_cuda()

    # -------------------------
    # Pass 2: Evaluate day-by-day
    # -------------------------
    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset(path, cfg, is_train=False)
            if ds is None:
                print(f"[{arch}][eval {file_idx}/{len(eval_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)

            for xb, yb in loader:
                xb_gpu = xb.to(DEEP_DEVICE, non_blocking=True)

                if amp_enabled:
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        logits = model(xb_gpu)
                else:
                    logits = model(xb_gpu)

                preds = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64, copy=False)
                y_true = yb.numpy().astype(np.int64, copy=False)

                batch_n = int(y_true.shape[0])
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows_seen[h] += batch_n
                    eval_class_counts[h] += np.bincount(y_true[:, i], minlength=3)

                del xb, xb_gpu, yb, logits, preds, y_true

            print(
                f"[{arch}][eval {file_idx}/{len(eval_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows}"
            )

            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    runtime_s = time.time() - start_t

    weights_path = os.path.join(cfg['weights_dir'], f'{arch}_weights.pt')
    torch.save(
        {
            'architecture': arch,
            'state_dict': model.state_dict(),
            'input_dim': len(DEEP_RAW_LOB_10_COLS),
            'horizons': horizons,
            'seq_len': int(cfg['seq_len']),
            'alpha': float(cfg['alpha']),
            'device_trained': str(DEEP_DEVICE),
        },
        weights_path,
    )

    run_meta = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep',
        'architecture': arch,
        'tickers': tickers,
        'horizons': horizons,
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'train_config': {
            'epochs': int(cfg['epochs']),
            'batch_size': int(cfg['batch_size']),
            'lr': float(cfg['lr']),
            'weight_decay': float(cfg['weight_decay']),
            'grad_clip': float(cfg['grad_clip']),
            'amp': bool(cfg.get('amp', True)),
            'device': str(DEEP_DEVICE),
        },
        'sampling': {
            'train_file_fraction': float(cfg['train_file_fraction']),
            'base_subsample': int(cfg['base_subsample']),
            'high_pressure_subsample': int(cfg['high_pressure_subsample']),
            'critical_pressure_subsample': int(cfg['critical_pressure_subsample']),
            'max_rows_per_day_train': int(cfg['max_rows_per_day_train']),
            'max_rows_per_day_eval': int(cfg['max_rows_per_day_eval']),
            'max_files_per_ticker': int(cfg['max_files_per_ticker']),
        },
        'files': {
            'train_files': len(train_files),
            'eval_files': len(eval_files),
            'days_fitted': int(total_train_days),
        },
        'train_rows_seen': {f'h{h}': int(train_rows_seen[h]) for h in horizons},
        'eval_rows_seen': {f'h{h}': int(eval_rows_seen[h]) for h in horizons},
        'train_class_counts': {f'h{h}': [int(x) for x in train_class_counts[h].tolist()] for h in horizons},
        'eval_class_counts': {f'h{h}': [int(x) for x in eval_class_counts[h].tolist()] for h in horizons},
        'test_metrics': metrics,
        'model_path': weights_path,
        'runtime_seconds': round(runtime_s, 2),
    }

    results_path = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming.json')
    with open(results_path, 'w') as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] saved weights -> {weights_path}")
    print(f"[{arch}] saved metrics -> {results_path}")
    deep_mem(f"{arch} end")

    del model, optimizer, scaler
    _deep_cleanup_cuda()

    return run_meta


def run_all_deep_models_day_by_day(config: dict | None = None) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    deep_set_seed(int(cfg['seed']))
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    all_runs: Dict[str, dict] = {}
    all_results_paths: Dict[str, str] = {}

    t0 = time.time()
    for arch in cfg['run_architectures']:
        run_meta = train_one_deep_architecture(arch, cfg)
        all_runs[arch] = run_meta
        all_results_paths[arch] = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming.json')
        _deep_cleanup_cuda()

    summary = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_multi_arch',
        'architectures': list(cfg['run_architectures']),
        'horizons': list(cfg['horizons']),
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'results_paths': all_results_paths,
        'runtime_seconds': round(time.time() - t0, 2),
    }

    summary_path = os.path.join(cfg['results_dir'], 'deep_models_day_streaming_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n' + '=' * 88)
    print('Deep day-by-day run complete')
    print(f"Summary saved -> {summary_path}")

    return {
        'summary_path': summary_path,
        'summary': summary,
        'runs': all_runs,
    }


# Kick off all 3 requested architectures.
deep_runs = run_all_deep_models_day_by_day()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Stability Patch + Full-Data Training (All files per ticker)
# ══════════════════════════════════════════════════════════════════════════════

# 1) Use all available files per ticker (you have 20 per stock now).
if 'CONFIG' in globals():
    CONFIG['max_files_per_ticker'] = 0

if 'DEEP_CONFIG' in globals():
    DEEP_CONFIG.update({
        'max_files_per_ticker': 0,      # 0 => no cap, use all available files
        'epochs': 2,
        'batch_size': 64,
        'lr': 1e-4,
        'weight_decay': 5e-5,
        'grad_clip': 1.0,
        'amp': False,                   # disable mixed precision for numerical stability
        'label_smoothing': 0.02,
        'result_suffix': '_stable_v2',
    })

print('Patched CONFIG max_files_per_ticker:', CONFIG.get('max_files_per_ticker', 'n/a') if 'CONFIG' in globals() else 'n/a')
print('Patched DEEP_CONFIG max_files_per_ticker:', DEEP_CONFIG.get('max_files_per_ticker', 'n/a') if 'DEEP_CONFIG' in globals() else 'n/a')
print('Patched DEEP_CONFIG epochs/lr/amp:', DEEP_CONFIG.get('epochs'), DEEP_CONFIG.get('lr'), DEEP_CONFIG.get('amp'))


class StableDayWindowDataset(Dataset):
    """Day dataset with per-day z-score normalization to prevent NaN blow-ups."""

    def __init__(self, raw: np.ndarray, starts: np.ndarray, labels: np.ndarray, seq_len: int):
        raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)

        self.raw = raw
        self.starts = starts.astype(np.int64, copy=False)
        self.labels = labels.astype(np.int64, copy=False)
        self.seq_len = int(seq_len)

        feat_mean = raw.mean(axis=0, dtype=np.float64).astype(np.float32)
        feat_std = raw.std(axis=0, dtype=np.float64).astype(np.float32)
        feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)

        self.feat_mean = feat_mean
        self.feat_std = feat_std

    def __len__(self) -> int:
        return int(self.starts.size)

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        x = self.raw[s : s + self.seq_len]

        x = (x - self.feat_mean) / self.feat_std
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x = np.clip(x, -10.0, 10.0)

        y = self.labels[idx]
        return torch.from_numpy(x.astype(np.float32, copy=False)), torch.from_numpy(y)


def _deep_build_day_dataset_stable(
    parquet_path: str,
    cfg: dict,
    is_train: bool,
) -> Tuple[StableDayWindowDataset | None, DayStats | None]:
    if 'make_fixed_threshold_classification_labels' not in globals():
        raise RuntimeError('make_fixed_threshold_classification_labels is not defined. Run labels cell first.')

    horizons = list(cfg['horizons'])
    seq_len = int(cfg['seq_len'])
    alpha = float(cfg['alpha'])

    step = _deep_choose_subsample(cfg)
    max_rows = _deep_choose_max_rows(cfg, is_train=is_train)

    df = pd.read_parquet(parquet_path, columns=DEEP_RAW_LOB_10_COLS)
    raw = np.ascontiguousarray(df.to_numpy(dtype=np.float32, copy=False))
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

    y_float = make_fixed_threshold_classification_labels(
        df,
        horizons=horizons,
        alpha=alpha,
        use_smoothing=True,
    ).to_numpy(dtype=np.float32, copy=False)

    max_h = int(max(horizons))
    valid_end = len(df) - max_h

    if valid_end <= seq_len:
        del df, raw, y_float
        gc.collect()
        return None, None

    labels = y_float[seq_len - 1 : valid_end]
    valid_mask = ~np.isnan(labels).any(axis=1)
    starts = np.flatnonzero(valid_mask).astype(np.int64, copy=False)

    if step > 1:
        starts = starts[::step]

    if max_rows > 0:
        starts = starts[:max_rows]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts
        gc.collect()
        return None, None

    y_day = labels[starts].astype(np.int64, copy=False)
    class_mask = ((y_day >= 0) & (y_day <= 2)).all(axis=1)
    starts = starts[class_mask]
    y_day = y_day[class_mask]

    if starts.size == 0:
        del df, raw, y_float, labels, valid_mask, starts, y_day, class_mask
        gc.collect()
        return None, None

    ds = StableDayWindowDataset(raw=raw, starts=starts, labels=y_day, seq_len=seq_len)
    stats = DayStats(
        ticker=os.path.basename(os.path.dirname(parquet_path)),
        file_name=os.path.basename(parquet_path),
        rows_raw=int(len(df)),
        rows_kept=int(starts.size),
        step=int(step),
        max_rows=int(max_rows),
    )

    del df, y_float, labels, valid_mask, starts, y_day, class_mask
    gc.collect()
    return ds, stats


def _deep_class_weights_stable(labels: np.ndarray) -> List[np.ndarray]:
    weights: List[np.ndarray] = []
    for i in range(labels.shape[1]):
        y = labels[:, i]
        # Add pseudocounts to avoid unstable swings when a class is sparse in a single day.
        counts = np.bincount(y.astype(np.int64, copy=False), minlength=3).astype(np.float64) + 10.0
        w = counts.sum() / (3.0 * counts)
        w = np.clip(w, 0.5, 3.0)
        weights.append(w.astype(np.float32))
    return weights


def _deep_multihorizon_loss_stable(
    logits: torch.Tensor,
    targets: torch.Tensor,
    weight_tensors: List[torch.Tensor] | None = None,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    losses = []
    for i in range(logits.shape[1]):
        w = None if weight_tensors is None else weight_tensors[i]
        losses.append(
            F.cross_entropy(
                logits[:, i, :],
                targets[:, i],
                weight=w,
                label_smoothing=float(label_smoothing),
            )
        )
    return torch.stack(losses).mean()


def train_one_deep_architecture_stable(arch: str, config: dict | None = None) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    horizons = list(cfg['horizons'])

    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    tickers = _deep_resolve_tickers(cfg)
    if not tickers:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg['data_dir'],
        tickers,
        int(cfg['max_files_per_ticker']),
    )
    if not files_by_ticker:
        raise FileNotFoundError('No parquet files found for selected tickers.')

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker,
        float(cfg['train_file_fraction']),
    )

    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
    )

    amp_enabled = bool(cfg.get('amp', False)) and DEEP_DEVICE.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    confusion = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    train_rows_seen = {h: 0 for h in horizons}
    eval_rows_seen = {h: 0 for h in horizons}
    train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    total_train_days = 0
    bad_batches_total = 0
    start_t = time.time()

    print('\n' + '=' * 88)
    print(f"Stable training architecture: {arch}")
    print(
        f"Device={DEEP_DEVICE} | train_files={len(train_files)} eval_files={len(eval_files)} "
        f"| max_files_per_ticker={cfg['max_files_per_ticker']}"
    )
    deep_mem(f"{arch} stable start")

    # -------------------------
    # Pass 1: Train day-by-day
    # -------------------------
    for epoch in range(int(cfg['epochs'])):
        model.train()
        print(f"\n[{arch}] epoch {epoch + 1}/{int(cfg['epochs'])}")

        for file_idx, (ticker, path) in enumerate(train_files, start=1):
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=True)
            if ds is None:
                print(f"[{arch}][train {file_idx}/{len(train_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=True)
            day_weights_np = _deep_class_weights_stable(ds.labels)
            day_weights = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in day_weights_np]

            loss_sum = 0.0
            good_batches = 0
            bad_batches = 0

            for xb, yb in loader:
                xb = xb.to(DEEP_DEVICE, non_blocking=True)
                yb = yb.to(DEEP_DEVICE, non_blocking=True, dtype=torch.long)

                xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)

                optimizer.zero_grad(set_to_none=True)

                if amp_enabled:
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        logits = model(xb)
                        logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                        loss = _deep_multihorizon_loss_stable(
                            logits,
                            yb,
                            day_weights,
                            label_smoothing=float(cfg.get('label_smoothing', 0.0)),
                        )

                    if not torch.isfinite(loss):
                        bad_batches += 1
                        del xb, yb, logits, loss
                        continue

                    scaler.scale(loss).backward()
                    if float(cfg['grad_clip']) > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['grad_clip']))
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    logits = model(xb)
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                    loss = _deep_multihorizon_loss_stable(
                        logits,
                        yb,
                        day_weights,
                        label_smoothing=float(cfg.get('label_smoothing', 0.0)),
                    )

                    if not torch.isfinite(loss):
                        bad_batches += 1
                        del xb, yb, logits, loss
                        continue

                    loss.backward()
                    if float(cfg['grad_clip']) > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg['grad_clip']))
                    optimizer.step()

                loss_sum += float(loss.item())
                good_batches += 1

                del xb, yb, logits, loss

            bad_batches_total += bad_batches

            if good_batches == 0:
                print(
                    f"[{arch}][train {file_idx}/{len(train_files)}] "
                    f"{ticker}/{stats.file_name} had no finite batches; day skipped"
                )
                del loader, ds, day_weights, day_weights_np
                _deep_cleanup_cuda()
                continue

            for i, h in enumerate(horizons):
                train_rows_seen[h] += int(ds.labels.shape[0])
                train_class_counts[h] += np.bincount(ds.labels[:, i], minlength=3)

            total_train_days += 1
            day_loss = loss_sum / max(1, good_batches)
            print(
                f"[{arch}][train {file_idx}/{len(train_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows} loss={day_loss:.4f} "
                f"good_batches={good_batches} bad_batches={bad_batches}"
            )

            del loader, ds, day_weights, day_weights_np
            _deep_cleanup_cuda()

    # -------------------------
    # Pass 2: Evaluate day-by-day
    # -------------------------
    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=False)
            if ds is None:
                print(f"[{arch}][eval {file_idx}/{len(eval_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)

            for xb, yb in loader:
                xb_gpu = xb.to(DEEP_DEVICE, non_blocking=True)
                xb_gpu = torch.nan_to_num(xb_gpu, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)

                logits = model(xb_gpu)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                preds = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64, copy=False)
                y_true = yb.numpy().astype(np.int64, copy=False)

                batch_n = int(y_true.shape[0])
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows_seen[h] += batch_n
                    eval_class_counts[h] += np.bincount(y_true[:, i], minlength=3)

                del xb, xb_gpu, yb, logits, preds, y_true

            print(
                f"[{arch}][eval {file_idx}/{len(eval_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows}"
            )

            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    runtime_s = time.time() - start_t

    suffix = str(cfg.get('result_suffix', '_stable_v2'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    weights_path = os.path.join(cfg['weights_dir'], f'{arch}{suffix}_weights.pt')
    torch.save(
        {
            'architecture': arch,
            'state_dict': model.state_dict(),
            'input_dim': len(DEEP_RAW_LOB_10_COLS),
            'horizons': horizons,
            'seq_len': int(cfg['seq_len']),
            'alpha': float(cfg['alpha']),
            'device_trained': str(DEEP_DEVICE),
            'stable_training': True,
        },
        weights_path,
    )

    run_meta = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_stable',
        'architecture': arch,
        'tickers': tickers,
        'horizons': horizons,
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'train_config': {
            'epochs': int(cfg['epochs']),
            'batch_size': int(cfg['batch_size']),
            'lr': float(cfg['lr']),
            'weight_decay': float(cfg['weight_decay']),
            'grad_clip': float(cfg['grad_clip']),
            'amp': bool(cfg.get('amp', False)),
            'label_smoothing': float(cfg.get('label_smoothing', 0.0)),
            'device': str(DEEP_DEVICE),
        },
        'sampling': {
            'train_file_fraction': float(cfg['train_file_fraction']),
            'base_subsample': int(cfg['base_subsample']),
            'high_pressure_subsample': int(cfg['high_pressure_subsample']),
            'critical_pressure_subsample': int(cfg['critical_pressure_subsample']),
            'max_rows_per_day_train': int(cfg['max_rows_per_day_train']),
            'max_rows_per_day_eval': int(cfg['max_rows_per_day_eval']),
            'max_files_per_ticker': int(cfg['max_files_per_ticker']),
        },
        'files': {
            'train_files': len(train_files),
            'eval_files': len(eval_files),
            'days_fitted': int(total_train_days),
        },
        'train_rows_seen': {f'h{h}': int(train_rows_seen[h]) for h in horizons},
        'eval_rows_seen': {f'h{h}': int(eval_rows_seen[h]) for h in horizons},
        'train_class_counts': {f'h{h}': [int(x) for x in train_class_counts[h].tolist()] for h in horizons},
        'eval_class_counts': {f'h{h}': [int(x) for x in eval_class_counts[h].tolist()] for h in horizons},
        'bad_batches_total': int(bad_batches_total),
        'test_metrics': metrics,
        'model_path': weights_path,
        'runtime_seconds': round(runtime_s, 2),
    }

    results_path = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')
    with open(results_path, 'w') as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] stable saved weights -> {weights_path}")
    print(f"[{arch}] stable saved metrics -> {results_path}")
    deep_mem(f"{arch} stable end")

    del model, optimizer, scaler
    _deep_cleanup_cuda()

    return run_meta


def run_all_deep_models_day_by_day_stable(config: dict | None = None) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    deep_set_seed(int(cfg['seed']))
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    all_runs: Dict[str, dict] = {}
    all_results_paths: Dict[str, str] = {}

    t0 = time.time()
    for arch in cfg['run_architectures']:
        run_meta = train_one_deep_architecture_stable(arch, cfg)
        all_runs[arch] = run_meta

        suffix = str(cfg.get('result_suffix', '_stable_v2'))
        if suffix and not suffix.startswith('_'):
            suffix = '_' + suffix
        all_results_paths[arch] = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')

        _deep_cleanup_cuda()

    suffix = str(cfg.get('result_suffix', '_stable_v2'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    summary = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_multi_arch_stable',
        'architectures': list(cfg['run_architectures']),
        'horizons': list(cfg['horizons']),
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'results_paths': all_results_paths,
        'runtime_seconds': round(time.time() - t0, 2),
    }

    summary_path = os.path.join(cfg['results_dir'], f'deep_models_day_streaming_summary{suffix}.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n' + '=' * 88)
    print('Stable deep day-by-day run complete')
    print(f"Summary saved -> {summary_path}")

    return {
        'summary_path': summary_path,
        'summary': summary,
        'runs': all_runs,
    }


# Run this stable pipeline (all files, NaN guards, normalized inputs).
deep_runs_stable = run_all_deep_models_day_by_day_stable()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Fair RF (76/20 split) + Deep Early Stopping (macro-F1)
# ══════════════════════════════════════════════════════════════════════════════

import copy

def _mean_macro_f1(metrics: dict, horizons: list[int]) -> float:
    vals = [float(metrics.get(f'h{h}_f1_macro', 0.0)) for h in horizons]
    return float(np.mean(vals)) if vals else 0.0


def run_rf_fullsplit_fair(config_override: dict | None = None) -> tuple[dict, str]:
    """Run RF with the same split/sampling regime used by stable deep runs."""
    if 'main_day_by_day' not in globals():
        raise RuntimeError('main_day_by_day is not defined. Run the RF streaming cell first.')

    if 'CONFIG' not in globals():
        raise RuntimeError('CONFIG is not defined. Run the RF setup cell first.')

    cfg = dict(CONFIG)

    deep_cfg = dict(DEEP_CONFIG) if 'DEEP_CONFIG' in globals() else {}
    cfg.update({
        'train_file_fraction': float(deep_cfg.get('train_file_fraction', cfg.get('train_file_fraction', 0.8))),
        'max_files_per_ticker': 0,
        'base_subsample': int(deep_cfg.get('base_subsample', cfg.get('base_subsample', 8))),
        'high_pressure_subsample': int(deep_cfg.get('high_pressure_subsample', cfg.get('high_pressure_subsample', 16))),
        'critical_pressure_subsample': int(deep_cfg.get('critical_pressure_subsample', cfg.get('critical_pressure_subsample', 32))),
        'max_rows_per_day_train': int(deep_cfg.get('max_rows_per_day_train', cfg.get('max_rows_per_day_train', 12000))),
        'max_rows_per_day_eval': int(deep_cfg.get('max_rows_per_day_eval', cfg.get('max_rows_per_day_eval', 16000))),
    })

    if config_override:
        cfg.update(config_override)

    tickers = resolve_tickers(cfg)
    files_by_ticker = _collect_files_by_ticker(cfg['data_dir'], tickers, int(cfg['max_files_per_ticker']))
    train_files, eval_files = _split_train_eval_files(files_by_ticker, float(cfg['train_file_fraction']))

    trees_per_day = int(cfg.get('trees_per_day', 4))
    cfg['trees_per_day'] = trees_per_day
    cfg['n_estimators'] = max(int(cfg.get('n_estimators', 0)), len(train_files) * trees_per_day)

    print('\n' + '=' * 88)
    print('RF fair full-split run')
    print(
        f"tickers={tickers} | train_files={len(train_files)} eval_files={len(eval_files)} "
        f"trees_per_day={cfg['trees_per_day']} n_estimators={cfg['n_estimators']}"
    )

    rf_meta = main_day_by_day(cfg)

    fair_results_path = os.path.join(cfg['results_dir'], 'rf_wallbridge_results_fullsplit_7620.json')
    with open(fair_results_path, 'w') as f:
        json.dump(rf_meta, f, indent=2)

    print(f"RF fair results saved -> {fair_results_path}")
    return rf_meta, fair_results_path


def _deep_train_epoch_stable(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    cfg: dict,
    train_files: list[tuple[str, str]],
    horizons: list[int],
    epoch_idx: int,
    total_epochs: int,
    amp_enabled: bool,
    label_smoothing: float,
    grad_clip: float,
    arch: str,
) -> dict:
    model.train()
    print(f"\n[{arch}] epoch {epoch_idx}/{total_epochs}")

    train_rows_seen = {h: 0 for h in horizons}
    train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    bad_batches_total = 0
    fitted_days = 0
    day_losses: list[float] = []

    for file_idx, (ticker, path) in enumerate(train_files, start=1):
        ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=True)
        if ds is None:
            print(f"[{arch}][train {file_idx}/{len(train_files)}] skipped empty day {os.path.basename(path)}")
            continue

        loader = _deep_make_loader(ds, cfg, is_train=True)
        day_weights_np = _deep_class_weights_stable(ds.labels)
        day_weights = [torch.tensor(w, dtype=torch.float32, device=DEEP_DEVICE) for w in day_weights_np]

        loss_sum = 0.0
        good_batches = 0
        bad_batches = 0

        for xb, yb in loader:
            xb = xb.to(DEEP_DEVICE, non_blocking=True)
            yb = yb.to(DEEP_DEVICE, non_blocking=True, dtype=torch.long)

            xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)
            optimizer.zero_grad(set_to_none=True)

            if amp_enabled:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(xb)
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                    loss = _deep_multihorizon_loss_stable(
                        logits,
                        yb,
                        day_weights,
                        label_smoothing=label_smoothing,
                    )

                if not torch.isfinite(loss):
                    bad_batches += 1
                    del xb, yb, logits, loss
                    continue

                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(xb)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                loss = _deep_multihorizon_loss_stable(
                    logits,
                    yb,
                    day_weights,
                    label_smoothing=label_smoothing,
                )

                if not torch.isfinite(loss):
                    bad_batches += 1
                    del xb, yb, logits, loss
                    continue

                loss.backward()
                if grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            loss_sum += float(loss.item())
            good_batches += 1

            del xb, yb, logits, loss

        bad_batches_total += bad_batches

        if good_batches == 0:
            print(
                f"[{arch}][train {file_idx}/{len(train_files)}] "
                f"{ticker}/{stats.file_name} had no finite batches; day skipped"
            )
            del loader, ds, day_weights, day_weights_np
            _deep_cleanup_cuda()
            continue

        fitted_days += 1
        day_loss = loss_sum / max(1, good_batches)
        day_losses.append(day_loss)

        for i, h in enumerate(horizons):
            train_rows_seen[h] += int(ds.labels.shape[0])
            train_class_counts[h] += np.bincount(ds.labels[:, i], minlength=3)

        print(
            f"[{arch}][train {file_idx}/{len(train_files)}] "
            f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
            f"max_rows={stats.max_rows} loss={day_loss:.4f} "
            f"good_batches={good_batches} bad_batches={bad_batches}"
        )

        del loader, ds, day_weights, day_weights_np
        _deep_cleanup_cuda()

    return {
        'train_rows_seen': train_rows_seen,
        'train_class_counts': train_class_counts,
        'bad_batches': int(bad_batches_total),
        'fitted_days': int(fitted_days),
        'avg_train_loss': float(np.mean(day_losses)) if day_losses else float('nan'),
    }


def _deep_eval_epoch_stable(
    model: nn.Module,
    cfg: dict,
    eval_files: list[tuple[str, str]],
    horizons: list[int],
    arch: str,
) -> dict:
    confusion = {h: np.zeros((3, 3), dtype=np.int64) for h in horizons}
    eval_rows_seen = {h: 0 for h in horizons}
    eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    model.eval()
    with torch.no_grad():
        for file_idx, (ticker, path) in enumerate(eval_files, start=1):
            ds, stats = _deep_build_day_dataset_stable(path, cfg, is_train=False)
            if ds is None:
                print(f"[{arch}][eval {file_idx}/{len(eval_files)}] skipped empty day {os.path.basename(path)}")
                continue

            loader = _deep_make_loader(ds, cfg, is_train=False)

            for xb, yb in loader:
                xb_gpu = xb.to(DEEP_DEVICE, non_blocking=True)
                xb_gpu = torch.nan_to_num(xb_gpu, nan=0.0, posinf=0.0, neginf=0.0).clamp(-10.0, 10.0)

                logits = model(xb_gpu)
                logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                preds = logits.argmax(dim=2).detach().cpu().numpy().astype(np.int64, copy=False)
                y_true = yb.numpy().astype(np.int64, copy=False)

                batch_n = int(y_true.shape[0])
                for i, h in enumerate(horizons):
                    _deep_update_confusion(confusion[h], y_true[:, i], preds[:, i])
                    eval_rows_seen[h] += batch_n
                    eval_class_counts[h] += np.bincount(y_true[:, i], minlength=3)

                del xb, xb_gpu, yb, logits, preds, y_true

            print(
                f"[{arch}][eval {file_idx}/{len(eval_files)}] "
                f"{ticker}/{stats.file_name} rows={stats.rows_kept} sub={stats.step} "
                f"max_rows={stats.max_rows}"
            )

            del loader, ds
            _deep_cleanup_cuda()

    metrics: Dict[str, float] = {}
    for h in horizons:
        metrics.update(_deep_metrics_from_confusion(confusion[h], h))

    return {
        'metrics': metrics,
        'confusion': confusion,
        'eval_rows_seen': eval_rows_seen,
        'eval_class_counts': eval_class_counts,
    }


def train_one_deep_architecture_stable_earlystop(
    arch: str,
    config: dict | None = None,
    max_epochs: int = 6,
    patience: int = 2,
    min_delta: float = 1e-4,
) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    cfg['max_files_per_ticker'] = 0
    cfg['epochs'] = 1

    horizons = list(cfg['horizons'])
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    tickers = _deep_resolve_tickers(cfg)
    if not tickers:
        raise FileNotFoundError(f"No tickers found in {cfg['data_dir']}")

    files_by_ticker = _deep_collect_files_by_ticker(
        cfg['data_dir'],
        tickers,
        int(cfg['max_files_per_ticker']),
    )
    if not files_by_ticker:
        raise FileNotFoundError('No parquet files found for selected tickers.')

    train_files, eval_files = _deep_split_train_eval_files(
        files_by_ticker,
        float(cfg['train_file_fraction']),
    )

    model = build_deep_model(
        arch=arch,
        input_dim=len(DEEP_RAW_LOB_10_COLS),
        horizon_count=len(horizons),
        num_classes=3,
    ).to(DEEP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
    )

    amp_enabled = bool(cfg.get('amp', False)) and DEEP_DEVICE.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    grad_clip = float(cfg.get('grad_clip', 1.0))
    label_smoothing = float(cfg.get('label_smoothing', 0.0))

    start_t = time.time()
    best_score = -1.0
    best_epoch = 0
    best_metrics: dict = {}
    best_eval_rows_seen = {h: 0 for h in horizons}
    best_eval_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}
    best_state_dict: dict | None = None

    no_improve = 0
    epoch_history: list[dict] = []

    total_bad_batches = 0
    total_train_days = 0
    agg_train_rows_seen = {h: 0 for h in horizons}
    agg_train_class_counts = {h: np.zeros(3, dtype=np.int64) for h in horizons}

    print('\n' + '=' * 88)
    print(f"Stable early-stop training architecture: {arch}")
    print(
        f"Device={DEEP_DEVICE} | train_files={len(train_files)} eval_files={len(eval_files)} "
        f"| max_epochs={max_epochs} patience={patience}"
    )
    deep_mem(f"{arch} stable-es start")

    for epoch in range(1, int(max_epochs) + 1):
        train_info = _deep_train_epoch_stable(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            cfg=cfg,
            train_files=train_files,
            horizons=horizons,
            epoch_idx=epoch,
            total_epochs=int(max_epochs),
            amp_enabled=amp_enabled,
            label_smoothing=label_smoothing,
            grad_clip=grad_clip,
            arch=arch,
        )

        total_bad_batches += int(train_info['bad_batches'])
        total_train_days += int(train_info['fitted_days'])
        for h in horizons:
            agg_train_rows_seen[h] += int(train_info['train_rows_seen'][h])
            agg_train_class_counts[h] += train_info['train_class_counts'][h]

        eval_info = _deep_eval_epoch_stable(
            model=model,
            cfg=cfg,
            eval_files=eval_files,
            horizons=horizons,
            arch=arch,
        )

        metrics = eval_info['metrics']
        score = _mean_macro_f1(metrics, horizons)
        improved = score > (best_score + float(min_delta))

        epoch_history.append({
            'epoch': int(epoch),
            'mean_macro_f1': float(score),
            'avg_train_loss': float(train_info['avg_train_loss']),
            'bad_batches': int(train_info['bad_batches']),
            'improved': bool(improved),
        })

        print(
            f"[{arch}] epoch={epoch} mean_macro_f1={score:.6f} "
            f"best={best_score:.6f} improved={improved}"
        )

        if improved:
            best_score = float(score)
            best_epoch = int(epoch)
            best_metrics = dict(metrics)
            best_eval_rows_seen = {h: int(eval_info['eval_rows_seen'][h]) for h in horizons}
            best_eval_class_counts = {h: eval_info['eval_class_counts'][h].copy() for h in horizons}
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= int(patience):
            print(f"[{arch}] early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

        _deep_cleanup_cuda()

    if best_state_dict is None:
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = int(len(epoch_history)) if epoch_history else 1
        best_metrics = dict(eval_info['metrics']) if 'eval_info' in locals() else {}
        best_score = _mean_macro_f1(best_metrics, horizons)

    model.load_state_dict(best_state_dict)

    suffix = str(cfg.get('result_suffix', '_stable_es'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    weights_path = os.path.join(cfg['weights_dir'], f'{arch}{suffix}_weights.pt')
    torch.save(
        {
            'architecture': arch,
            'state_dict': model.state_dict(),
            'input_dim': len(DEEP_RAW_LOB_10_COLS),
            'horizons': horizons,
            'seq_len': int(cfg['seq_len']),
            'alpha': float(cfg['alpha']),
            'device_trained': str(DEEP_DEVICE),
            'stable_training': True,
            'early_stopping': {
                'max_epochs': int(max_epochs),
                'patience': int(patience),
                'min_delta': float(min_delta),
                'best_epoch': int(best_epoch),
                'best_mean_macro_f1': float(best_score),
                'epochs_ran': int(len(epoch_history)),
            },
        },
        weights_path,
    )

    runtime_s = time.time() - start_t
    run_meta = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_stable_earlystop',
        'architecture': arch,
        'tickers': tickers,
        'horizons': horizons,
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'train_config': {
            'max_epochs': int(max_epochs),
            'patience': int(patience),
            'min_delta': float(min_delta),
            'batch_size': int(cfg['batch_size']),
            'lr': float(cfg['lr']),
            'weight_decay': float(cfg['weight_decay']),
            'grad_clip': float(cfg['grad_clip']),
            'amp': bool(cfg.get('amp', False)),
            'label_smoothing': float(cfg.get('label_smoothing', 0.0)),
            'device': str(DEEP_DEVICE),
        },
        'sampling': {
            'train_file_fraction': float(cfg['train_file_fraction']),
            'base_subsample': int(cfg['base_subsample']),
            'high_pressure_subsample': int(cfg['high_pressure_subsample']),
            'critical_pressure_subsample': int(cfg['critical_pressure_subsample']),
            'max_rows_per_day_train': int(cfg['max_rows_per_day_train']),
            'max_rows_per_day_eval': int(cfg['max_rows_per_day_eval']),
            'max_files_per_ticker': int(cfg['max_files_per_ticker']),
        },
        'files': {
            'train_files': len(train_files),
            'eval_files': len(eval_files),
            'days_fitted': int(total_train_days),
        },
        'early_stopping': {
            'best_epoch': int(best_epoch),
            'best_mean_macro_f1': float(best_score),
            'epochs_ran': int(len(epoch_history)),
        },
        'epoch_history': epoch_history,
        'train_rows_seen': {f'h{h}': int(agg_train_rows_seen[h]) for h in horizons},
        'eval_rows_seen': {f'h{h}': int(best_eval_rows_seen[h]) for h in horizons},
        'train_class_counts': {f'h{h}': [int(x) for x in agg_train_class_counts[h].tolist()] for h in horizons},
        'eval_class_counts': {f'h{h}': [int(x) for x in best_eval_class_counts[h].tolist()] for h in horizons},
        'bad_batches_total': int(total_bad_batches),
        'test_metrics': best_metrics,
        'model_path': weights_path,
        'runtime_seconds': round(runtime_s, 2),
    }

    results_path = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')
    with open(results_path, 'w') as f:
        json.dump(run_meta, f, indent=2)

    print(f"[{arch}] stable-es saved weights -> {weights_path}")
    print(f"[{arch}] stable-es saved metrics -> {results_path}")
    deep_mem(f"{arch} stable-es end")

    del model, optimizer, scaler
    _deep_cleanup_cuda()

    return run_meta


def run_all_deep_models_day_by_day_stable_earlystop(
    config: dict | None = None,
    max_epochs: int = 6,
    patience: int = 2,
    min_delta: float = 1e-4,
) -> dict:
    cfg = dict(DEEP_CONFIG)
    if config is not None:
        cfg.update(config)

    cfg['max_files_per_ticker'] = 0
    cfg.setdefault('result_suffix', '_stable_es')

    deep_set_seed(int(cfg['seed']))
    os.makedirs(cfg['weights_dir'], exist_ok=True)
    os.makedirs(cfg['results_dir'], exist_ok=True)

    all_runs: Dict[str, dict] = {}
    all_results_paths: Dict[str, str] = {}

    t0 = time.time()
    for arch in cfg['run_architectures']:
        run_meta = train_one_deep_architecture_stable_earlystop(
            arch=arch,
            config=cfg,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
        )
        all_runs[arch] = run_meta

        suffix = str(cfg.get('result_suffix', '_stable_es'))
        if suffix and not suffix.startswith('_'):
            suffix = '_' + suffix
        all_results_paths[arch] = os.path.join(cfg['results_dir'], f'{arch}_results_day_streaming{suffix}.json')

        _deep_cleanup_cuda()

    suffix = str(cfg.get('result_suffix', '_stable_es'))
    if suffix and not suffix.startswith('_'):
        suffix = '_' + suffix

    summary = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'mode': 'day_by_day_streaming_deep_multi_arch_stable_earlystop',
        'architectures': list(cfg['run_architectures']),
        'horizons': list(cfg['horizons']),
        'seq_len': int(cfg['seq_len']),
        'alpha': float(cfg['alpha']),
        'early_stopping': {
            'max_epochs': int(max_epochs),
            'patience': int(patience),
            'min_delta': float(min_delta),
        },
        'results_paths': all_results_paths,
        'runtime_seconds': round(time.time() - t0, 2),
    }

    summary_path = os.path.join(cfg['results_dir'], f'deep_models_day_streaming_summary{suffix}.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n' + '=' * 88)
    print('Stable deep early-stop run complete')
    print(f"Summary saved -> {summary_path}")

    return {
        'summary_path': summary_path,
        'summary': summary,
        'runs': all_runs,
    }


# -----------------------------
# Execute both requested steps
# -----------------------------
RUN_RF_FULLSPLIT = True
RUN_DEEP_EARLYSTOP = True

rf_full_meta = None
rf_full_results_path = None
deep_runs_stable_es = None

if RUN_RF_FULLSPLIT:
    rf_full_meta, rf_full_results_path = run_rf_fullsplit_fair()

if RUN_DEEP_EARLYSTOP:
    deep_es_cfg = {
        'result_suffix': '_stable_es',
        'max_files_per_ticker': 0,
        'amp': False,
        'label_smoothing': float(DEEP_CONFIG.get('label_smoothing', 0.02)),
    }
    deep_runs_stable_es = run_all_deep_models_day_by_day_stable_earlystop(
        config=deep_es_cfg,
        max_epochs=6,
        patience=2,
        min_delta=1e-4,
    )

if (rf_full_meta is not None) and (deep_runs_stable_es is not None):
    horizons = list(DEEP_CONFIG['horizons'])
    rf_score = _mean_macro_f1(rf_full_meta.get('test_metrics', {}), horizons)
    print('\n' + '-' * 88)
    print(f"RF fair mean macro-F1: {rf_score:.6f}")

    for arch, run_meta in deep_runs_stable_es['runs'].items():
        deep_score = _mean_macro_f1(run_meta.get('test_metrics', {}), horizons)
        best_epoch = int(run_meta.get('early_stopping', {}).get('best_epoch', 0))
        print(f"{arch}: mean macro-F1={deep_score:.6f} | best_epoch={best_epoch}")

    print('-' * 88)
    print(f"RF fair results path: {rf_full_results_path}")
    print(f"Deep early-stop summary: {deep_runs_stable_es['summary_path']}")

In [ ]:
# Cell 12: Pre-flight paths + prerequisite symbol check
from pathlib import Path

if "DATA_DIR" not in globals():
    DATA_DIR = str(Path.cwd() / "data" / "processed")
if "WEIGHTS_DIR" not in globals():
    WEIGHTS_DIR = str(Path.cwd() / "model_weights")
if "RESULTS_DIR" not in globals():
    RESULTS_DIR = str(Path.cwd() / "results")

Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR    = {DATA_DIR}")
print(f"WEIGHTS_DIR = {WEIGHTS_DIR}")
print(f"RESULTS_DIR = {RESULTS_DIR}")

required_symbols = [
    "make_fixed_threshold_classification_labels",
    "main_day_by_day",
    "build_deep_model",
    "run_all_deep_models_day_by_day_stable_earlystop",
]
missing = [name for name in required_symbols if name not in globals()]

if missing:
    print("\nMissing symbols:")
    for name in missing:
        print(f"  - {name}")
    print(
        "\nRun these previous cells first, in order:\n"
        "1) Label generation cell (Optimized multi-horizon label generation)\n"
        "2) RF streaming cell (main_day_by_day)\n"
        "3) Deep setup cell (Cell 7)\n"
        "4) Deep model definitions (Cell 8)\n"
        "5) Stable deep patch cell (Cell 10)\n"
        "6) Early-stop cell (Cell 11)"
    )
else:
    print("\nAll required symbols are available. You can run the multi-run cells now.")

In [ ]:
# Cell 13: Read existing result files and baseline leaderboard
import json
import numpy as np
import pandas as pd


def mean_macro_f1(metrics: dict, horizons=(10, 20, 50, 100)) -> float:
    vals = [float(metrics.get(f"h{h}_f1_macro", np.nan)) for h in horizons]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.mean(vals)) if vals else float("nan")


def safe_load_json(path: Path):
    try:
        with path.open("r") as f:
            return json.load(f)
    except Exception:
        return None


results_dir_path = Path(RESULTS_DIR)
result_files = sorted(results_dir_path.glob("*results*.json"))

baseline_rows = []
for p in result_files:
    meta = safe_load_json(p)
    if not isinstance(meta, dict):
        continue

    metrics = meta.get("test_metrics", {})
    if not isinstance(metrics, dict) or not metrics:
        continue

    model_name = meta.get("architecture")
    if model_name is None and "rf_wallbridge" in p.name:
        model_name = "rf_wallbridge"
    if model_name is None:
        model_name = "unknown"

    baseline_rows.append(
        {
            "file": p.name,
            "model": model_name,
            "mode": str(meta.get("mode", "")),
            "timestamp": str(meta.get("timestamp", "")),
            "mean_macro_f1": mean_macro_f1(metrics),
            "h10_f1_macro": float(metrics.get("h10_f1_macro", np.nan)),
            "h20_f1_macro": float(metrics.get("h20_f1_macro", np.nan)),
            "h50_f1_macro": float(metrics.get("h50_f1_macro", np.nan)),
            "h100_f1_macro": float(metrics.get("h100_f1_macro", np.nan)),
        }
    )

baseline_df = pd.DataFrame(baseline_rows)
if baseline_df.empty:
    print("No per-model test_metrics JSON files found yet.")
else:
    baseline_df = baseline_df.sort_values(["mean_macro_f1", "timestamp"], ascending=[False, False]).reset_index(drop=True)
    display(baseline_df)

In [ ]:
# Cell 14: Multi-run experiment config (first step: repeat training)
MULTI_RUN_CONFIG = {
    "n_repeats": 3,
    "seed_start": 42,
    "seed_step": 11,
    "run_rf": True,
    "run_deep": True,
    "rf_config_override": {
        "max_files_per_ticker": 0,
        "train_file_fraction": 0.8,
        "trees_per_day": 4,
        "n_estimators": 304,
        "n_jobs": 1,
        "max_depth": 20,
        "min_samples_leaf": 20,
    },
    "deep_config_override": {
        "max_files_per_ticker": 0,
        "train_file_fraction": 0.8,
        "batch_size": 64,
        "lr": 1e-4,
        "weight_decay": 5e-5,
        "grad_clip": 1.0,
        "amp": False,
        "label_smoothing": 0.02,
        "max_epochs": 4,
        "patience": 2,
        "min_delta": 1e-4,
        "result_suffix_prefix": "_stable_es_repeat",
    },
}

print("MULTI_RUN_CONFIG loaded.")
print(pd.Series({k: v for k, v in MULTI_RUN_CONFIG.items() if k not in ("rf_config_override", "deep_config_override")}))
print("RF override:", MULTI_RUN_CONFIG["rf_config_override"])
print("Deep override:", MULTI_RUN_CONFIG["deep_config_override"])

In [ ]:
# Cell 15: Execute repeated training runs for all notebook models
import time

required_runtime_symbols = []
if bool(MULTI_RUN_CONFIG.get("run_rf", True)):
    required_runtime_symbols.append("main_day_by_day")
if bool(MULTI_RUN_CONFIG.get("run_deep", True)):
    required_runtime_symbols.append("run_all_deep_models_day_by_day_stable_earlystop")

missing_runtime = [s for s in required_runtime_symbols if s not in globals()]
if missing_runtime:
    raise RuntimeError(
        "Missing required symbols: "
        + ", ".join(missing_runtime)
        + ". Run the prerequisite cells listed in Cell 12 first."
    )

horizons = list(globals().get("DEFAULT_HORIZONS", [10, 20, 50, 100]))
results_dir_path = Path(RESULTS_DIR)
results_dir_path.mkdir(parents=True, exist_ok=True)

seed_list = [
    int(MULTI_RUN_CONFIG["seed_start"] + i * MULTI_RUN_CONFIG["seed_step"])
    for i in range(int(MULTI_RUN_CONFIG["n_repeats"]))
]

multi_run_rows = []
multi_run_manifest = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "horizons": horizons,
    "config": MULTI_RUN_CONFIG,
    "runs": [],
}

for run_idx, seed in enumerate(seed_list, start=1):
    run_tag = f"repeat_{run_idx:02d}_seed_{seed}"
    print("\n" + "=" * 88)
    print(f"Starting {run_tag}")

    run_record = {
        "run_idx": int(run_idx),
        "seed": int(seed),
        "run_tag": run_tag,
        "artifacts": {},
    }

    if bool(MULTI_RUN_CONFIG.get("run_rf", True)):
        rf_cfg = dict(MULTI_RUN_CONFIG["rf_config_override"])
        rf_cfg["random_state"] = int(seed)

        t0 = time.time()
        rf_meta = main_day_by_day(rf_cfg)
        rf_runtime = time.time() - t0
        rf_score = mean_macro_f1(rf_meta.get("test_metrics", {}), horizons=horizons)

        rf_out = results_dir_path / f"rf_wallbridge_results_{run_tag}.json"
        with rf_out.open("w") as f:
            json.dump(rf_meta, f, indent=2)

        run_record["artifacts"]["rf_wallbridge"] = str(rf_out)
        run_record["rf_mean_macro_f1"] = float(rf_score)
        run_record["rf_runtime_seconds"] = round(float(rf_runtime), 2)

        multi_run_rows.append(
            {
                "run_tag": run_tag,
                "model": "rf_wallbridge",
                "mean_macro_f1": float(rf_score),
                "runtime_seconds": round(float(rf_runtime), 2),
                "result_file": rf_out.name,
            }
        )

    if bool(MULTI_RUN_CONFIG.get("run_deep", True)):
        deep_cfg = dict(MULTI_RUN_CONFIG["deep_config_override"])
        max_epochs = int(deep_cfg.pop("max_epochs", 4))
        patience = int(deep_cfg.pop("patience", 2))
        min_delta = float(deep_cfg.pop("min_delta", 1e-4))
        suffix_prefix = str(deep_cfg.pop("result_suffix_prefix", "_stable_es_repeat"))

        deep_cfg["seed"] = int(seed)
        deep_cfg["result_suffix"] = f"{suffix_prefix}_{run_idx:02d}"

        t0 = time.time()
        deep_bundle = run_all_deep_models_day_by_day_stable_earlystop(
            config=deep_cfg,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
        )
        deep_runtime = time.time() - t0

        run_record["artifacts"]["deep_summary"] = str(deep_bundle.get("summary_path", ""))
        run_record["deep_runtime_seconds"] = round(float(deep_runtime), 2)

        for arch, meta in deep_bundle.get("runs", {}).items():
            score = mean_macro_f1(meta.get("test_metrics", {}), horizons=horizons)
            out_path = results_dir_path / f"{arch}_results_{run_tag}.json"
            with out_path.open("w") as f:
                json.dump(meta, f, indent=2)

            run_record["artifacts"][arch] = str(out_path)
            multi_run_rows.append(
                {
                    "run_tag": run_tag,
                    "model": arch,
                    "mean_macro_f1": float(score),
                    "runtime_seconds": round(float(meta.get("runtime_seconds", np.nan)), 2),
                    "result_file": out_path.name,
                }
            )

    multi_run_manifest["runs"].append(run_record)
    print(f"Completed {run_tag}")

multi_run_df = pd.DataFrame(multi_run_rows)
if multi_run_df.empty:
    print("No runs were executed. Check MULTI_RUN_CONFIG.")
else:
    multi_run_df = multi_run_df.sort_values(["mean_macro_f1", "run_tag"], ascending=[False, True]).reset_index(drop=True)
    display(multi_run_df)

timestamp_slug = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
leaderboard_json = results_dir_path / f"multi_run_leaderboard_{timestamp_slug}.json"
manifest_json = results_dir_path / f"multi_run_manifest_{timestamp_slug}.json"

if not multi_run_df.empty:
    multi_run_df.to_json(leaderboard_json, orient="records", indent=2)

with manifest_json.open("w") as f:
    json.dump(multi_run_manifest, f, indent=2)

print("\nSaved:")
if not multi_run_df.empty:
    print(f"  leaderboard: {leaderboard_json}")
print(f"  manifest:    {manifest_json}")